# OvaScope — Parasite Egg Image Classification Pipeline

## Section 1 — Project introduction

**OvaScope** is an end-to-end machine-learning system that classifies microscope images of
human intestinal parasite eggs into **11 categories**.

**Why it matters.** Intestinal parasites are diagnosed by a trained technician who looks for eggs in
stool samples under a microscope. The work is slow, needs expertise, and gets tiring with large
sample numbers. An automatic classifier could help with triage and training. In this project it is a
research prototype only.

**Dataset.** [Chula-ParasiteEgg-11](https://zenodo.org/records/15866298) (Chulalongkorn University and
collaborators; ICIP 2022 challenge). It has 11,000 training photos and 2,200 test photos from several
microscopes, cameras and smartphones, with COCO bounding boxes around every egg.

**What the model does.** It takes an image **already cropped around one egg** and outputs a probability
for each of the 11 classes.

**What it does not do.** It does **not** find or count eggs in a full microscope photo (that would need
object detection). It is **not** a clinically validated diagnostic tool, and its confidence is a model
score, not a guarantee.

**Outputs of this notebook**
* verified single-egg crop dataset (rebuilt here from the official files)
* exploratory analysis of three image features
* a baseline CNN and a MobileNetV2 transfer-learning model, trained and compared on validation data
* per-epoch tracking of accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC and log loss
  (training and validation sets)
* validation and final held-out test evaluation with the same seven metrics, overall and for each class,
  plus confusion matrices and ROC / precision-recall curves
* saved `.keras` and load-tested `.h5` model, class mapping, metadata
* `ovascope_training_results.zip` with all results

## Section 2 — Learning objectives and workflow

```
Official data (Zenodo) ─► Crops (one egg each) ─► Verification ─► Exploration (EDA)
        ─► Preprocessing (resize, augment, batch) ─► Training (baseline CNN, MobileNetV2)
        ─► Validation-based model selection ─► Final test evaluation ─► Saving ─► Prediction
        ─► Retraining readiness ─► Export
```

Rules followed throughout:
1. **Three separate sets.** *train* fits weights. *validation* drives early stopping and model selection.
   *test* is used **once**, at the end, on the selected model only.
2. **All pipeline code is visible here.** Every function used from data collection to model saving
   is printed in this notebook, copied from the project modules in `src/`. The API and the retraining
   job import the same functions from `src/`, so training and serving cannot disagree.
3. **Only real results.** Every number shown below is produced by the cells in this notebook.

## Section 3 — Environment setup

The notebook runs in three environments:

| Environment | Project code | Data | Outputs |
|---|---|---|---|
| Local Jupyter | the repository (auto-detected) | `data/processed` (crops) and `data/raw` (official zips) | the repository (`models/`, `artifacts/`) |
| Google Colab | clone, or upload/unzip the project, or Google Drive | crops ZIP uploaded or on Drive | `/content/ovascope_outputs` (or Drive) |
| Kaggle | project attached as an input dataset | crops ZIP / folder attached as input | `/kaggle/working/ovascope_outputs` |

The next cell holds **all user settings**. `ENVIRONMENT = "auto"` detects the platform; set it by hand
if detection is wrong. No path is hard-coded to one person's home folder.

In [ ]:
import os
import platform
import sys
from pathlib import Path

# ---------------- Settings you may change ----------------
ENVIRONMENT = "auto"       # "auto" | "local" | "colab" | "kaggle"
PROJECT_DIR = None         # folder containing src/ and configs/ (None = auto-detect)
DATA_DIR = None            # folder with crops/, manifest.csv, class_map.json (None = auto-detect)
RAW_DATA_DIR = None        # folder with the official Zenodo files (only needed to rebuild crops)
OUTPUT_DIR = None          # writable folder for models and results (None = environment default)
REBUILD_CROPS = True       # rebuild crops from the official archives and prove they are identical
QUICK_RUN = False          # True = 1-2 epochs per stage, only to check that everything runs
USE_GOOGLE_DRIVE = False   # Colab: mount Google Drive (data and/or results live there)
PROJECT_REPO_URL = ""      # optional: git URL to clone the project code if it is not present
# ----------------------------------------------------------


def detect_environment():
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    return "local"


ENV = detect_environment() if ENVIRONMENT == "auto" else ENVIRONMENT
print("Environment:", ENV)
print("Python:", sys.version.split()[0], "|", platform.platform())
print("Working directory:", Path.cwd())

In [ ]:
if ENV == "colab" and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


def looks_like_project(folder: Path) -> bool:
    return (folder / "src" / "preprocessing.py").exists() and (folder / "configs" / "config.yaml").exists()


def find_project_dir():
    if PROJECT_DIR:
        return Path(PROJECT_DIR)
    candidates = [Path.cwd(), Path.cwd().parent]
    if ENV == "colab":
        candidates += [Path("/content/ovascope-MLOP"), Path("/content/drive/MyDrive/ovascope-MLOP")]
    if ENV == "kaggle":
        candidates += sorted(Path("/kaggle/input").glob("*")) + sorted(Path("/kaggle/input").glob("*/*"))
        candidates += [Path("/kaggle/working/ovascope-MLOP")]
    for folder in candidates:
        if looks_like_project(folder):
            return folder.resolve()
    if PROJECT_REPO_URL:
        target = Path("/content" if ENV == "colab" else "/kaggle/working" if ENV == "kaggle" else ".") / "ovascope-MLOP"
        os.system(f"git clone --depth 1 {PROJECT_REPO_URL} {target}")
        if looks_like_project(target):
            return target.resolve()
    raise FileNotFoundError(
        "Project code (src/, configs/) not found. Local: start Jupyter inside the repository. "
        "Colab: upload and unzip the project to /content/ovascope-MLOP (or set PROJECT_REPO_URL). "
        "Kaggle: attach the project as an input dataset.")


PROJECT = find_project_dir()
sys.path.insert(0, str(PROJECT))
print("Project directory:", PROJECT)

## Section 4 — Dependency management

Libraries used: **TensorFlow/Keras** (models), **NumPy / pandas** (arrays, tables), **Pillow** (image
files), **scikit-learn** (metrics), **Matplotlib** (plots), **PyYAML** (configuration).
The next cell installs a package **only if it cannot be imported**. Colab and Kaggle already ship
TensorFlow, so normally nothing is installed there. Exact tested versions are listed in
`requirements.txt`.

In [ ]:
import importlib
import subprocess

REQUIRED = {"tensorflow": "tensorflow", "numpy": "numpy", "pandas": "pandas", "PIL": "pillow",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib", "yaml": "pyyaml"}
for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installing missing package: {package}")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])
        if result.returncode != 0:
            raise RuntimeError(f"Could not install {package}; install it manually and restart the kernel.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

# Standard-library and scientific imports used by the pipeline code copied into this notebook
import csv
import hashlib
import io
import json
import random
import re
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from datetime import datetime, timezone

import yaml
from PIL import ImageOps
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, precision_recall_curve,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

### About the code in this notebook

Every function used from data collection to model saving is **defined in this notebook**, so the whole
pipeline can be read from top to bottom. Each code cell says which project file it was copied from
(`src/config.py`, `src/preprocessing.py`, `src/model.py`, `src/registry.py`, `src/evaluation.py`,
`src/train.py`, `src/prediction.py`, `src/retrain.py`). The API, the Streamlit app and the retraining job
import the same functions from `src/`.

Only two small things differ from `src/`, because a notebook has no `__file__` and its functions can
only reach worker processes in certain ways:
* `find_project_root()` returns the project folder found above, not the parent of `src/`;
* parallel image processing goes through `_worker_pool()`. It uses worker processes with the *fork* start
  method (Linux: local, Colab and Kaggle) and falls back to threads elsewhere. The results are identical
  either way.

The first functions set the random seeds and describe the hardware.

In [ ]:
# --- copied from src/train.py (environment) ---
def set_seeds(seed: int) -> None:
    """Seed Python, NumPy and TensorFlow. Some GPU kernels (e.g. cuDNN convolution
    backprop) remain non-deterministic, so repeated GPU runs can differ slightly."""
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def configure_gpu() -> list[str]:
    """Let TensorFlow grow GPU memory as needed instead of grabbing all of it."""
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass  # already initialised
    return [g.name for g in gpus]


def describe_environment() -> dict:
    gpus = tf.config.list_physical_devices("GPU")
    gpu_names = []
    for gpu in gpus:
        try:
            gpu_names.append(tf.config.experimental.get_device_details(gpu).get("device_name", gpu.name))
        except Exception:
            gpu_names.append(gpu.name)
    import PIL
    import sklearn
    return {
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0], "platform": platform.platform(),
        "tensorflow": tf.__version__, "keras": tf.keras.__version__ if hasattr(tf.keras, "__version__") else "",
        "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "pillow": PIL.__version__, "cpu_count": os.cpu_count(),
        "accelerator": "GPU" if gpus else "CPU", "gpus": gpu_names,
    }


configure_gpu()
environment_info = describe_environment()
pd.Series(environment_info).to_frame("value")

If `accelerator` shows **CPU**, everything still works, but training is much slower. Use
`QUICK_RUN = True` to check the pipeline first. On Colab choose *Runtime → Change runtime type → GPU*;
on Kaggle switch the accelerator on in the notebook settings.

## Section 5 — Dataset location and configuration

All settings (image size, epochs, learning rates, split fractions…) live in **`configs/config.yaml`**, and
`load_config()` (below) reads them. Paths can be redirected with environment variables, which is how
the same code runs on Colab and Kaggle:

* **Local**: crops in `data/processed`, official archives in `data/raw`.
* **Colab**: upload `chula_parasite_egg11_crops.zip` (or keep it on Drive). The next cell finds and unzips it.
* **Kaggle**: attach the crops ZIP or folder as an input dataset (read-only, under `/kaggle/input`); outputs go to
  `/kaggle/working`.

The large dataset is never downloaded again if it is already present.

In [ ]:
# --- copied from src/config.py ---
# Reads configs/config.yaml and resolves every path. Environment variables can redirect paths,
# which is how the same code runs locally, on Colab and on Kaggle.
ENV_OVERRIDES = {
    "processed_dir": "OVASCOPE_PROCESSED_DIR",
    "models_dir": "OVASCOPE_MODELS_DIR",
    "artifacts_dir": "OVASCOPE_ARTIFACTS_DIR",
    "database": "OVASCOPE_DB",
    "uploads_dir": "OVASCOPE_UPLOADS_DIR",
    "raw_dir": "OVASCOPE_RAW_DIR",
    "extracted_dir": "OVASCOPE_EXTRACTED_DIR",
}


def find_project_root() -> Path:
    # Notebook version: the project folder detected in Section 3 (src/config.py uses __file__ instead).
    if os.environ.get("OVASCOPE_ROOT"):
        return Path(os.environ["OVASCOPE_ROOT"]).resolve()
    return PROJECT


def load_config(config_path: str | Path | None = None) -> dict:
    root = find_project_root()
    config_path = Path(config_path) if config_path else root / "configs" / "config.yaml"
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    resolved = {}
    for key, value in cfg["paths"].items():
        value = os.environ.get(ENV_OVERRIDES.get(key, ""), value)
        path = Path(value)
        resolved[key] = path if path.is_absolute() else root / path
    cfg["paths"] = resolved
    cfg["paths"]["root"] = root
    cfg["paths"]["crops_dir"] = resolved["processed_dir"] / "crops"
    return cfg

In [ ]:
import zipfile

if OUTPUT_DIR is None:
    OUTPUT_DIR = {"local": PROJECT,
                  "colab": Path("/content/drive/MyDrive/ovascope_outputs") if USE_GOOGLE_DRIVE else Path("/content/ovascope_outputs"),
                  "kaggle": Path("/kaggle/working/ovascope_outputs")}[ENV]
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    if DATA_DIR:
        return Path(DATA_DIR)
    if ENV == "local" and (PROJECT / "data" / "processed" / "manifest.csv").exists():
        return PROJECT / "data" / "processed"
    search_roots = {"colab": [Path("/content"), Path("/content/drive/MyDrive")],
                    "kaggle": [Path("/kaggle/input")], "local": [PROJECT / "data"]}[ENV]
    for root in search_roots:
        for manifest in root.glob("**/manifest.csv"):
            if (manifest.parent / "crops").is_dir():
                return manifest.parent
    for root in search_roots:   # a crops ZIP: extract it to the writable output folder
        for archive in root.glob("**/chula_parasite_egg11_crops.zip"):
            target = OUTPUT_DIR / "data" / "processed"
            if not (target / "manifest.csv").exists():
                print("Extracting", archive)
                zipfile.ZipFile(archive).extractall(target)
            return target
    return OUTPUT_DIR / "data" / "processed"   # empty: crops will be built from the raw archives

def find_raw_dir():
    if RAW_DATA_DIR:
        return Path(RAW_DATA_DIR)
    roots = {"local": [PROJECT / "data" / "raw"], "colab": [Path("/content"), Path("/content/drive/MyDrive")],
             "kaggle": [Path("/kaggle/input")]}[ENV]
    for root in roots:
        for archive in root.glob("**/Chula-ParasiteEgg-11.zip"):
            return archive.parent
    return None

PROCESSED_DIR = find_data_dir()
RAW_DIR = find_raw_dir()
os.environ["OVASCOPE_PROCESSED_DIR"] = str(PROCESSED_DIR)
os.environ["OVASCOPE_MODELS_DIR"] = str(OUTPUT_DIR / "models")
os.environ["OVASCOPE_ARTIFACTS_DIR"] = str(OUTPUT_DIR / "artifacts")
os.environ["OVASCOPE_DB"] = str(OUTPUT_DIR / "db" / "ovascope.db")
if RAW_DIR:
    os.environ["OVASCOPE_RAW_DIR"] = str(RAW_DIR)
    if ENV != "local":
        os.environ["OVASCOPE_EXTRACTED_DIR"] = str(OUTPUT_DIR / "data" / "extracted")

cfg = load_config()
if QUICK_RUN:
    cfg["training"]["baseline"]["epochs"] = 2
    cfg["training"]["transfer"].update(head_epochs=1, fine_tune_epochs=1)
SEED = cfg["project"]["random_seed"]
IMAGE_SIZE = cfg["training"]["image_size"]
BATCH_SIZE = cfg["training"]["batch_size"]
PLOTS = cfg["paths"]["artifacts_dir"] / "plots"
METRICS = cfg["paths"]["artifacts_dir"] / "metrics"
PLOTS.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"setting": ["environment", "processed data", "raw archives", "outputs", "image size", "batch size",
                          "random seed", "quick run"],
              "value": [ENV, PROCESSED_DIR, RAW_DIR, OUTPUT_DIR, IMAGE_SIZE, BATCH_SIZE, SEED, QUICK_RUN]})

### Rebuilding the crop dataset from the official files

The official dataset was made for **object detection**: each photo comes with boxes around its eggs.
Our task is **classification**, so `build_crop_dataset` (below) turns every box into its own image:

1. **Verify** the downloaded archives against the MD5 checksums published on Zenodo.
2. **Pad** each box by 10% per side, so the whole shell plus a little background is kept.
3. **Make it square** around its centre. Later resizing to 224×224 then cannot stretch the egg.
4. **Clamp** the square inside the photo (no artificial padding pixels).
5. **Split** the official training photos 85/15 into train/validation. The split is stratified by class and
   **grouped by photo**, so eggs from one photo never land in different splits. The official test set
   (labelled by `test_labels_200.json`) is the held-out test set.
6. Write `manifest.csv` (one row per crop, with its source photo and box) and `class_map.json`.

The next three cells hold the complete data-preparation code. When `REBUILD_CROPS = True` and the official files are available, the next cell first fingerprints
(MD5) every existing crop. It then rebuilds everything and checks that the result is **byte-for-byte identical**.
This shows that the crop dataset is fully reproducible from the official source.

#### Data collection code (1/3): checksums, archives, class map and COCO annotations

In [ ]:
# --- copied from src/preprocessing.py (Part A: dataset preparation) ---
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


DISPLAY_NAMES = {
    "Opisthorchis viverrine": "Opisthorchis viverrini",
    "Paragonimus spp": "Paragonimus spp.",
}


MANIFEST_FIELDS = ["crop_id", "crop_path", "split", "class_id", "class_name", "folder",
                   "source_image", "annotation_id", "bbox_xywh", "crop_box_ltrb"]


def md5_of_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_archives(raw_dir: Path, expected_md5: dict) -> dict:
    """Return {filename: "ok" | "missing" | "checksum mismatch"} for the official files."""
    status = {}
    for name, md5 in expected_md5.items():
        path = Path(raw_dir) / name
        if not path.exists():
            status[name] = "missing"
        else:
            status[name] = "ok" if md5_of_file(path) == md5 else "checksum mismatch"
    return status


def extract_archive(zip_path: Path, destination: Path) -> None:
    """Unzip, refusing any member that would land outside `destination`."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.namelist():
            target = (destination / member).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Unsafe path in archive: {member}")
        archive.extractall(destination)


def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def build_class_map(categories: list[dict]) -> list[dict]:
    """Class map that keeps the official ids and names, plus display names and folders."""
    class_map = []
    for category in sorted(categories, key=lambda c: c["id"]):
        display_name = DISPLAY_NAMES.get(category["name"], category["name"])
        class_map.append({
            "class_id": category["id"],
            "original_name": category["name"],
            "display_name": display_name,
            "folder": f"{category['id']:02d}_{slugify(display_name)}",
        })
    return class_map


def load_coco(path: Path):
    data = json.loads(Path(path).read_text())
    images = {image["id"]: image for image in data["images"]}
    annotations = defaultdict(list)
    for annotation in data["annotations"]:
        annotations[annotation["image_id"]].append(annotation)
    return data["categories"], images, annotations

#### Data collection code (2/3): square crops and the photo-grouped train/validation split

`_worker_pool` is the notebook's replacement for `ProcessPoolExecutor` (see *About the code in this notebook*).

In [ ]:
def _worker_pool(workers):
    """Pool for parallel image work. Functions defined in a notebook reach worker processes only through
    the 'fork' start method (Linux: local, Colab, Kaggle); elsewhere use threads (same results, slower)."""
    import multiprocessing
    if "fork" in multiprocessing.get_all_start_methods():
        return ProcessPoolExecutor(max_workers=max(1, workers), mp_context=multiprocessing.get_context("fork"))
    return ThreadPoolExecutor(max_workers=max(1, workers))


# --- copied from src/preprocessing.py ---
def square_crop_box(bbox_xywh, margin: float, image_width: int, image_height: int):
    """Pad a COCO [x, y, w, h] box by `margin` per side, make it square, keep it inside the image."""
    x, y, w, h = bbox_xywh
    center_x, center_y = x + w / 2, y + h / 2
    side = max(w, h) * (1 + 2 * margin)
    side = round(min(side, image_width, image_height))
    left = round(min(max(center_x - side / 2, 0), image_width - side))
    top = round(min(max(center_y - side / 2, 0), image_height - side))
    return left, top, left + side, top + side


def split_train_val(images: dict, annotations: dict, val_fraction: float, seed: int) -> dict:
    """Assign each *source image* to train or val.

    Stratified by the image's class and grouped by source image, so all crops
    from one photo share a split (no photo appears in both train and val).
    """
    image_ids_by_class = defaultdict(list)
    for image_id in images:
        if annotations.get(image_id):
            image_ids_by_class[annotations[image_id][0]["category_id"]].append(image_id)
    rng = random.Random(seed)
    split_of_image = {}
    for class_id in sorted(image_ids_by_class):
        image_ids = sorted(image_ids_by_class[class_id])
        rng.shuffle(image_ids)
        n_val = round(len(image_ids) * val_fraction)
        for position, image_id in enumerate(image_ids):
            split_of_image[image_id] = "val" if position < n_val else "train"
    return split_of_image


def index_image_files(folder: Path) -> dict:
    return {p.name: p for p in Path(folder).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS}


def _crop_one_source_image(job):
    """Crop every annotated egg in one source photo (runs in a worker process)."""
    source_path, annotations, split, margin, out_dir, class_folders, class_names, file_name = job
    rows = []
    with Image.open(source_path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        for annotation in annotations:
            box = square_crop_box(annotation["bbox"], margin, image.width, image.height)
            folder = class_folders[annotation["category_id"]]
            crop_id = f"{Path(source_path).stem}_ann{annotation['id']}"
            destination = Path(out_dir) / "crops" / split / folder / f"{crop_id}.jpg"
            destination.parent.mkdir(parents=True, exist_ok=True)
            image.crop(box).save(destination, quality=95)
            rows.append({
                "crop_id": crop_id,
                "crop_path": str(destination.relative_to(out_dir)),
                "split": split,
                "class_id": annotation["category_id"],
                "class_name": class_names[annotation["category_id"]],
                "folder": folder,
                "source_image": file_name,
                "annotation_id": annotation["id"],
                "bbox_xywh": json.dumps(annotation["bbox"]),
                "crop_box_ltrb": json.dumps(list(box)),
            })
    return rows


def build_crop_dataset(train_json: Path, train_images: Path, test_json: Path, test_images: Path,
                       out_dir: Path, val_fraction: float = 0.15, margin: float = 0.10,
                       seed: int = 42, workers: int = 4) -> dict:
    """Create crops/{train,val,test}/<class>/, class_map.json, manifest.csv, summary.json.

    train/val come from the official training set; test is the official test set
    with its released labels (never used for training or model selection).
    The output is deterministic for a given seed, margin and Pillow version.
    """
    out_dir = Path(out_dir)
    categories, train_coco_images, train_annotations = load_coco(train_json)
    test_categories, test_coco_images, test_annotations = load_coco(test_json)
    if [c["name"] for c in categories] != [c["name"] for c in test_categories]:
        raise ValueError("Train and test annotation files disagree on the category list")

    class_map = build_class_map(categories)
    class_folders = {c["class_id"]: c["folder"] for c in class_map}
    class_names = {c["class_id"]: c["display_name"] for c in class_map}
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "class_map.json").write_text(json.dumps(class_map, indent=2))

    split_of_image = split_train_val(train_coco_images, train_annotations, val_fraction, seed)
    jobs, missing = [], []
    for coco_images, annotations, image_folder, split_for in [
        (train_coco_images, train_annotations, train_images, lambda i: split_of_image.get(i, "train")),
        (test_coco_images, test_annotations, test_images, lambda i: "test"),
    ]:
        files = index_image_files(image_folder)
        for image_id, image in coco_images.items():
            if not annotations.get(image_id):
                continue
            source_path = files.get(Path(image["file_name"]).name)
            if source_path is None:
                missing.append(image["file_name"])
                continue
            jobs.append((source_path, annotations[image_id], split_for(image_id), margin,
                         out_dir, class_folders, class_names, image["file_name"]))

    rows = []
    with _worker_pool(workers) as pool:
        for job_rows in pool.map(_crop_one_source_image, jobs, chunksize=32):
            rows.extend(job_rows)
    rows.sort(key=lambda r: (r["split"], r["class_id"], r["crop_id"]))

    with open(out_dir / "manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

    counts = Counter((r["split"], r["class_id"]) for r in rows)
    summary = {split: {class_folders[c]: counts[(split, c)] for c in sorted(class_folders)}
               for split in ("train", "val", "test")}
    summary["missing_images"] = missing
    summary["settings"] = {"val_fraction": val_fraction, "margin": margin, "seed": seed}
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

#### Data collection code (3/3): manifest, verification, protected test hashes and the end-to-end `prepare_dataset`

In [ ]:
# --- copied from src/preprocessing.py ---
def read_manifest(processed_dir: Path) -> list[dict]:
    with open(Path(processed_dir) / "manifest.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        row["class_id"] = int(row["class_id"])
    return rows


def load_class_map(path: Path) -> list[dict]:
    """Load class_map.json (accepts a file or the folder that contains it)."""
    path = Path(path)
    if path.is_dir():
        path = path / "class_map.json"
    class_map = json.loads(path.read_text())
    ids = [c["class_id"] for c in class_map]
    if ids != list(range(len(ids))):
        raise ValueError(f"class_map.json ids must be 0..N-1 in order, got {ids}")
    return class_map


def verify_crop_dataset(processed_dir: Path, check_decode: bool = True) -> dict:
    """Check that every crop in the manifest exists, decodes, and sits in the right folder."""
    processed_dir = Path(processed_dir)
    rows = read_manifest(processed_dir)
    class_map = load_class_map(processed_dir)
    folder_of = {c["class_id"]: c["folder"] for c in class_map}
    missing, corrupt, wrong_folder = [], [], []
    for row in rows:
        path = processed_dir / row["crop_path"]
        if not path.exists():
            missing.append(row["crop_path"])
            continue
        if path.parent.name != folder_of[row["class_id"]] or path.parent.parent.name != row["split"]:
            wrong_folder.append(row["crop_path"])
        if check_decode:
            try:
                with Image.open(path) as image:
                    image.verify()
            except Exception:
                corrupt.append(row["crop_path"])
    on_disk = {str(p.relative_to(processed_dir)) for p in (processed_dir / "crops").rglob("*.jpg")}
    untracked = sorted(on_disk - {r["crop_path"] for r in rows})
    counts = Counter((r["split"], r["class_id"]) for r in rows)
    return {
        "total_crops": len(rows),
        "per_split": dict(Counter(r["split"] for r in rows)),
        "per_split_class": {f"{s}/{folder_of[c]}": n for (s, c), n in sorted(counts.items())},
        "missing": missing, "corrupt": corrupt, "wrong_folder": wrong_folder,
        "untracked_files": untracked,
        "ok": not (missing or corrupt or wrong_folder or untracked),
    }


def prepare_dataset(cfg: dict, rebuild: bool = False, workers: int = 4, log=print) -> dict:
    """End-to-end data preparation used by the notebook and scripts/prepare_data.py.

    1. If crops already exist and verify cleanly (and rebuild=False), reuse them.
    2. Otherwise: verify archive checksums, unzip if needed, build crops.
    """
    paths, ds = cfg["paths"], cfg["dataset"]
    processed = paths["processed_dir"]
    if not rebuild and (processed / "manifest.csv").exists():
        report = verify_crop_dataset(processed, check_decode=False)
        if report["ok"]:
            log(f"Reusing existing crops in {processed} ({report['total_crops']} crops).")
            return report
        log("Existing crops failed verification; rebuilding.")

    extracted = paths["extracted_dir"]
    train_json = extracted / ds["train_json"]
    log("Verifying the official files against the Zenodo MD5 checksums...")
    status = verify_archives(paths["raw_dir"], ds["archives"])
    log(f"Archive check: {status}")
    archives_ok = all(v == "ok" for v in status.values())
    if not train_json.exists() and not archives_ok:
        raise FileNotFoundError(
            "Official archives missing or corrupt in "
            f"{paths['raw_dir']}. Download them from https://zenodo.org/records/{ds['zenodo_record']}")
    if not archives_ok:
        log("WARNING: archives could not be verified; using the already extracted files.")
    if not train_json.exists():
        for name in ds["archives"]:
            if name.endswith(".zip"):
                log(f"Extracting {name} ...")
                extract_archive(paths["raw_dir"] / name, extracted)

    if (processed / "crops").exists() and rebuild:
        import shutil
        shutil.rmtree(processed / "crops")
    log("Building crops (one image per annotated egg)...")
    summary = build_crop_dataset(
        train_json=train_json, train_images=extracted / ds["train_images"],
        test_json=paths["raw_dir"] / ds["test_json_in_raw"], test_images=extracted / ds["test_images"],
        out_dir=processed, val_fraction=ds["val_fraction"], margin=ds["crop_margin"],
        seed=cfg["project"]["random_seed"], workers=workers)
    if summary["missing_images"]:
        log(f"WARNING: {len(summary['missing_images'])} annotated images were not found on disk")
    report = verify_crop_dataset(processed)
    count = write_protected_hashes(processed, paths["models_dir"] / "protected_test_hashes.txt",
                                   [extracted / ds["test_images"]])
    log(f"Wrote {count} protected test-image hashes.")
    return report


def write_protected_hashes(processed_dir: Path, out_path: Path, extra_folders=()) -> int:
    """SHA-256 of every held-out test crop (and optionally the original test photos).

    The upload endpoint rejects any file with one of these hashes, so the
    protected test set cannot leak into retraining data."""
    hashes = set()
    test_files = [Path(processed_dir) / r["crop_path"] for r in read_manifest(processed_dir) if r["split"] == "test"]
    for folder in extra_folders:
        if Path(folder).exists():
            test_files += [p for p in Path(folder).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    for path in test_files:
        hashes.add(hashlib.sha256(path.read_bytes()).hexdigest())
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text("\n".join(sorted(hashes)) + "\n")
    return len(hashes)

In [ ]:
import hashlib
import time

def fingerprint_crops(processed_dir: Path) -> dict:
    return {str(p.relative_to(processed_dir)): hashlib.md5(p.read_bytes()).hexdigest()
            for p in sorted((processed_dir / "crops").rglob("*.jpg"))}

raw_available = bool(RAW_DIR) and (Path(RAW_DIR) / "Chula-ParasiteEgg-11.zip").exists()
if REBUILD_CROPS and raw_available:
    before = fingerprint_crops(PROCESSED_DIR) if (PROCESSED_DIR / "crops").exists() else {}
    start = time.time()
    report = prepare_dataset(cfg, rebuild=True, workers=os.cpu_count() or 2)
    print(f"Rebuilt {report['total_crops']} crops in {time.time() - start:.0f} s")
    after = fingerprint_crops(PROCESSED_DIR)
    if before:
        same = sum(before.get(k) == v for k, v in after.items())
        print(f"Previously existing crops: {len(before)} | rebuilt: {len(after)} | byte-identical: {same}")
        print("Reproducible:", same == len(before) == len(after))
else:
    if REBUILD_CROPS:
        print("Official archives not available here: reusing the prepared crops instead of rebuilding.")
    report = prepare_dataset(cfg, rebuild=False)

## Section 6 — Dataset verification

Before any modelling, check that every crop listed in the manifest exists, can be decoded, and sits in the
folder of its class and split. The notebook **stops** if anything is wrong.

In [ ]:
verification = verify_crop_dataset(PROCESSED_DIR, check_decode=True)
class_map = load_class_map(PROCESSED_DIR)
CLASS_NAMES = [c["display_name"] for c in class_map]
manifest = pd.DataFrame(read_manifest(PROCESSED_DIR))

print("Total crops:", verification["total_crops"], "| per split:", verification["per_split"])
print("Missing files:", len(verification["missing"]), "| corrupt files:", len(verification["corrupt"]),
      "| wrong folder:", len(verification["wrong_folder"]), "| untracked files:", len(verification["untracked_files"]))
if not verification["ok"]:
    raise RuntimeError("Dataset verification failed - fix the data before training.")
pd.DataFrame(class_map)

The class ids and the **official label strings are kept exactly as published**, including the dataset's
spelling *Opisthorchis viverrine*. Only the display name is normalised, to the usual species spelling
*Opisthorchis viverrini* (likewise *Paragonimus spp.*). The model always uses the numeric `class_id`,
never alphabetical folder order.

In [ ]:
counts = manifest.pivot_table(index="class_name", columns="split", values="crop_id", aggfunc="count")
counts = counts.loc[CLASS_NAMES, ["train", "val", "test"]]
counts.loc["TOTAL"] = counts.sum()
print("Source photos per split:", manifest.groupby("split")["source_image"].nunique().to_dict())
counts

**Interpretation.** All 13,259 crops exist, decode, and sit in the right class/split folder. They come from
13,200 source photos: 9,350 train, 1,650 validation and 2,200 test. Every class has 850–860 training,
150–151 validation and 199–209 test crops. Crop counts slightly exceed photo counts because 57 photos
contain two or three eggs. Next we explore what the images look like.

## Section 7 — Exploratory data analysis

Next we measure simple properties of **every crop** with `image_statistics` (code below):
* **brightness**: mean grey level, 0–255
* **contrast**: standard deviation of grey levels
* **sharpness**: variance of a Laplacian filter. It is high for crisp edges and low for blurry images.
* **saturation**: mean HSV saturation, i.e. how coloured the crop is versus grey
* **width / height / aspect ratio**

In [ ]:
# --- copied from src/preprocessing.py (image measurements for EDA) ---
def image_statistics(image: Image.Image) -> dict:
    """Simple, explainable measurements used for EDA and the dashboard.

    brightness  mean grey level (0-255)
    contrast    standard deviation of grey levels (RMS contrast)
    sharpness   variance of a Laplacian filter response - high for crisp edges,
                low for blurry / out-of-focus images (a texture/focus proxy)
    saturation  mean HSV saturation (0-255) - how coloured vs. grey the crop is
    """
    rgb = image.convert("RGB")
    grey = np.asarray(rgb.convert("L"), dtype=np.float32)
    laplacian = (-4 * grey[1:-1, 1:-1] + grey[:-2, 1:-1] + grey[2:, 1:-1]
                 + grey[1:-1, :-2] + grey[1:-1, 2:])
    red, green, blue = (np.asarray(rgb, dtype=np.float32)[..., i].mean() for i in range(3))
    return {
        "width": rgb.width, "height": rgb.height,
        "aspect_ratio": rgb.width / rgb.height,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "sharpness": float(laplacian.var()),
        "saturation": float(np.asarray(rgb.convert("HSV"), dtype=np.float32)[..., 1].mean()),
        "mean_red": float(red), "mean_green": float(green), "mean_blue": float(blue),
    }


def _statistics_of_file(path) -> dict:
    with Image.open(path) as image:
        return image_statistics(image)


def image_statistics_for_files(paths, workers: int = 4) -> list[dict]:
    """image_statistics() for many files, in parallel worker processes."""
    with _worker_pool(workers) as pool:
        return list(pool.map(_statistics_of_file, [str(p) for p in paths], chunksize=64))

In [ ]:
start = time.time()
stats = pd.DataFrame(image_statistics_for_files([PROCESSED_DIR / p for p in manifest["crop_path"]],
                                                workers=os.cpu_count() or 2))
stats = pd.concat([manifest[["crop_id", "split", "class_id", "class_name"]], stats], axis=1)
stats.to_csv(METRICS / "crop_image_stats_notebook.csv", index=False)
print(f"Measured {len(stats)} crops in {time.time() - start:.0f} s")
stats.describe().loc[["mean", "std", "min", "50%", "max"], ["width", "aspect_ratio", "brightness", "contrast",
                                                           "sharpness", "saturation"]].round(2)

### 7.1 Number of images per class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
counts.drop(index="TOTAL").plot.barh(stacked=True, ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xlabel("number of crops")
ax.set_ylabel("")
ax.invert_yaxis()
ax.set_title("Crops per class and split")
plt.tight_layout()
plt.savefig(PLOTS / "eda_class_distribution.png", dpi=120)
plt.show()
train_counts = counts.drop(index="TOTAL")["train"]
print(f"Training crops per class: min {train_counts.min()}, max {train_counts.max()}, "
      f"ratio {train_counts.max() / train_counts.min():.3f}")

**Interpretation.** The largest training class has only 1.2% more crops than the smallest (860 vs 850).
The dataset is **balanced**, so plain accuracy is meaningful and no class weights or resampling are needed.
Validation and test are balanced too, so macro and weighted averages will be almost identical.

### 7.2 Brightness (intensity) distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for split, part in stats.groupby("split"):
    axes[0].hist(part["brightness"], bins=50, alpha=0.5, density=True, label=split)
axes[0].set_xlabel("mean brightness (0-255)")
axes[0].set_ylabel("density")
axes[0].legend()
axes[0].set_title("Brightness by split")
stats.boxplot(column="brightness", by="class_name", ax=axes[1], vert=False, showfliers=False)
axes[1].set_title("Brightness by class")
axes[1].set_xlabel("mean brightness (0-255)")
plt.suptitle("")
plt.tight_layout()
plt.savefig(PLOTS / "eda_brightness.png", dpi=120)
plt.show()
stats.groupby("split")[["brightness", "contrast", "sharpness"]].median().round(1)

### 7.3 Image dimensions and aspect ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(stats["width"], bins=60, color="#4C72B0")
axes[0].axvline(IMAGE_SIZE, color="red", linestyle="--", label=f"model input {IMAGE_SIZE}px")
axes[0].set_xlabel("crop side length (px)")
axes[0].set_ylabel("crops")
axes[0].legend()
axes[0].set_title(f"Crop size (aspect ratio min {stats.aspect_ratio.min():.2f}, max {stats.aspect_ratio.max():.2f})")
stats.boxplot(column="width", by="class_name", ax=axes[1], vert=False, showfliers=False)
axes[1].set_xscale("log")
axes[1].set_title("Crop side length by class (log scale)")
axes[1].set_xlabel("pixels")
plt.suptitle("")
plt.tight_layout()
plt.savefig(PLOTS / "eda_dimensions.png", dpi=120)
plt.show()
print(f"Crops smaller than {IMAGE_SIZE}px (will be upsampled): {(stats.width < IMAGE_SIZE).mean():.1%}")

### 7.4 Representative parasite images (random crops, 6 per class)

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(11, 6, figsize=(9, 17))
for class_id, name in enumerate(CLASS_NAMES):
    rows = manifest[(manifest.class_id == class_id) & (manifest.split == "train")]
    for col, path in enumerate(rng.choice(rows["crop_path"].values, 6, replace=False)):
        axes[class_id, col].imshow(Image.open(PROCESSED_DIR / path))
        axes[class_id, col].axis("off")
    axes[class_id, 0].set_title(name, fontsize=9, loc="left")
plt.tight_layout()
plt.savefig(PLOTS / "eda_examples.png", dpi=100)
plt.show()

## Section 8 — Interpretation of three image features

To judge whether a feature carries **class information**, we use *η² (eta squared)*: the share of the
feature's variance explained by the class. 0 means unrelated to class; 1 means fully determined by it.

In [ ]:
def eta_squared(feature):
    group_mean = stats.groupby("class_name")[feature].transform("mean")
    return ((group_mean - stats[feature].mean()) ** 2).sum() / ((stats[feature] - stats[feature].mean()) ** 2).sum()

feature_table = pd.DataFrame({f: {"eta_squared": eta_squared(f), "median": stats[f].median(),
                                  "IQR": stats[f].quantile(0.75) - stats[f].quantile(0.25)}
                              for f in ["brightness", "contrast", "width", "sharpness", "saturation"]}).T.round(3)
feature_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
stats.boxplot(column="sharpness", by="class_name", ax=axes[0], vert=False, showfliers=False)
axes[0].set_xscale("log")
axes[0].set_title("Sharpness (Laplacian variance, log scale)")
stats.boxplot(column="saturation", by="class_name", ax=axes[1], vert=False, showfliers=False)
axes[1].set_title("Colour saturation (0-255)")
plt.suptitle("")
plt.tight_layout()
plt.savefig(PLOTS / "eda_texture_colour.png", dpi=120)
plt.show()

**Feature 1 — Brightness.** Mean brightness ranges from about 18 to 241 (median 151). The class explains only
**7.6%** of its variance: within every class there are dark and bright crops, so brightness reflects the
microscope, lamp and camera rather than the parasite. The held-out **test crops are darker** (median 138 vs
154 in train). *Implication:* a model relying on brightness would fail on other microscopes, and the test
set itself is shifted. We therefore use random brightness/contrast augmentation and expect some loss in
test performance.

**Feature 2 — Size and aspect ratio.** All crops are square by construction (aspect ratio exactly 1.0). Their
side length runs from 72 to 967 px, and 34% are smaller than the 224 px model input. Size is **strongly
class-dependent (η² = 0.755)**: *Fasciolopsis buski* crops have a median width of about 630 px, while
*Opisthorchis viverrini* is about 130 px. This matches real egg-size differences, but it also depends on
magnification. *Implication:* squaring before resizing protects egg shape. However, resizing everything to
224×224 removes absolute size, so the model must rely on shape and texture. Size-similar pairs such as the
two *Hymenolepis* species may therefore be harder to separate.

**Feature 3 — Texture/focus (Laplacian variance) and colour saturation.** Sharpness spans 1.7 to about
15,800 (log scale needed), and the class explains only **0.4%** of it. Focus quality depends on the
capture, not the species. Test crops are markedly **sharper** (median 1,483 vs 880). Saturation is
moderately class-related (**η² = 0.144**): some species' eggs are yellow-brown, others almost colourless.
*Limitation:* these are whole-crop averages. They mix egg and background debris, and a feature that differs
on average does not by itself let us classify single images. The overlapping boxes show none of them
separates the classes alone.

**Overall:** illumination and focus vary more *between capture setups* than *between classes*, and the
test set differs from the training set in both. This is a warning sign for generalisation that we check
again in Section 18.

## Section 9 — Image preprocessing

The code below (copied from `src/preprocessing.py`) is shared with the API:

| Step | What | Why |
|---|---|---|
| Loading | decode JPEG/PNG to RGB (`tf.io.decode_image`, or Pillow in the API) | consistent 3-channel input, even for greyscale or transparent PNG uploads |
| Resizing | square crop → 224×224, bilinear with antialiasing (`resize_image`) | fixed input size for the CNN; crops are already square, so shapes are not distorted |
| Normalisation | **inside the model**: `Rescaling(1/255)` (baseline) or `Rescaling(1/127.5, -1)` (MobileNetV2) | normalisation happens exactly once and travels with the saved model |
| Label encoding | integer `class_id` 0–10 from the manifest (sparse labels) | stable mapping, independent of folder ordering |
| Augmentation | random flips, rotation, small shift/zoom, ±10% brightness and contrast — **training only** | eggs have no fixed orientation under the microscope, and brightness varies between microscopes (Section 8) |
| Batching | `tf.data`: parallel decode → batch 32 → prefetch | keeps the GPU busy without loading all images into memory |

In [ ]:
# --- copied from src/preprocessing.py (image loading, resizing, augmentation) ---
def load_image_from_bytes(data: bytes) -> Image.Image:
    """Decode bytes into an RGB PIL image; raises ValueError for anything unreadable."""
    if not data:
        raise ValueError("Empty file")
    try:
        with Image.open(io.BytesIO(data)) as probe:
            probe.verify()                      # catches truncated / corrupt files
        image = Image.open(io.BytesIO(data))
        image = ImageOps.exif_transpose(image)
        return image.convert("RGB")
    except Exception as error:
        raise ValueError(f"Not a readable image: {error.__class__.__name__}") from None


def resize_image(image_uint8, image_size: int):
    """THE resize step. Input: HxWx3 uint8 (tensor or array). Output: float32 in [0, 255]."""
    import tensorflow as tf
    image = tf.image.resize(tf.cast(image_uint8, tf.float32), (image_size, image_size),
                            method="bilinear", antialias=True)
    return tf.clip_by_value(image, 0.0, 255.0)


def preprocess_pil_image(image: Image.Image, image_size: int) -> np.ndarray:
    """Serving path: PIL image -> (1, size, size, 3) float32 batch for model.predict."""
    array = np.asarray(image.convert("RGB"), dtype=np.uint8)
    return resize_image(array, image_size).numpy()[np.newaxis, ...]


def decode_image_bytes(contents):
    """Decode JPEG/PNG/BMP/GIF bytes to uint8 RGB inside tf.data.

    JPEGs are decoded with dct_method="INTEGER_ACCURATE", which reproduces
    Pillow's decoder exactly (verified: 0 pixel difference). TensorFlow's
    default fast decoder differs from Pillow by up to 16 grey levels, which
    would make training pixels differ from what the API (Pillow) sees.
    """
    import tensorflow as tf
    image = tf.cond(
        tf.io.is_jpeg(contents),
        lambda: tf.io.decode_jpeg(contents, channels=3, dct_method="INTEGER_ACCURATE"),
        lambda: tf.io.decode_image(contents, channels=3, expand_animations=False))
    image.set_shape([None, None, 3])
    return image


def _load_file(path, image_size):
    import tensorflow as tf
    return resize_image(decode_image_bytes(tf.io.read_file(path)), image_size)


def build_augmentation(seed: int = 42):
    """Training-only augmentation. Eggs have no canonical orientation under the
    microscope, so flips and rotations do not change the class; brightness and
    contrast changes mimic the different microscopes/cameras in the dataset."""
    import tensorflow as tf
    layers = tf.keras.layers
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=seed),
        layers.RandomRotation(0.5, fill_mode="reflect", seed=seed),
        layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=seed),
        layers.RandomZoom((-0.1, 0.1), fill_mode="reflect", seed=seed),
        layers.RandomBrightness(0.1, value_range=(0, 255), seed=seed),
        layers.RandomContrast(0.1, seed=seed),
    ], name="augmentation")

In [ ]:
example_path = PROCESSED_DIR / manifest[manifest.split == "train"].iloc[0]["crop_path"]
original = Image.open(example_path).convert("RGB")
resized = preprocess_pil_image(original, IMAGE_SIZE)          # (1, 224, 224, 3) float32 in [0, 255]
augment = build_augmentation(SEED)
augmented = [augment(resized, training=True).numpy()[0] for _ in range(4)]

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(original)
axes[0].set_title(f"original {original.width}x{original.height}")
axes[1].imshow(resized[0].astype("uint8"))
axes[1].set_title(f"resized {IMAGE_SIZE}x{IMAGE_SIZE}")
for i, image in enumerate(augmented):
    axes[i + 2].imshow(np.clip(image, 0, 255).astype("uint8"))
    axes[i + 2].set_title(f"augmented #{i + 1}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(PLOTS / "preprocessing_examples.png", dpi=110)
plt.show()
print("Model input: shape", resized.shape, "dtype", resized.dtype, "range", (resized.min(), resized.max()))
print("Inside MobileNetV2 the Rescaling layer maps it to", ((resized.min() / 127.5) - 1, (resized.max() / 127.5) - 1))

## Section 10 — Dataset preparation

Next we build the three `tf.data` pipelines and inspect one batch. **Leakage prevention:** we check that
no source photo contributes crops to more than one split, and the audit (`reports/dataset_audit.md`) found
0 exact and 0 near-duplicate photos across splits.

In [ ]:
# --- copied from src/preprocessing.py (tf.data pipelines) ---
def list_split(processed_dir: Path, split: str) -> tuple[list[str], list[int]]:
    """File paths and integer labels for one split, taken from the manifest (not folder order)."""
    rows = [r for r in read_manifest(processed_dir) if r["split"] == split]
    return [str(Path(processed_dir) / r["crop_path"]) for r in rows], [r["class_id"] for r in rows]


def make_dataset(file_paths, labels, image_size: int, batch_size: int, training: bool,
                 seed: int = 42, augmentation=None):
    """tf.data pipeline: read -> decode -> resize -> (shuffle + augment if training) -> batch -> prefetch."""
    import tensorflow as tf
    autotune = tf.data.AUTOTUNE
    dataset = tf.data.Dataset.from_tensor_slices((list(file_paths), list(labels)))
    if training:
        dataset = dataset.shuffle(len(file_paths), seed=seed, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda p, y: (_load_file(p, image_size), y), num_parallel_calls=autotune)
    dataset = dataset.batch(batch_size)
    if training:
        augmentation = augmentation or build_augmentation(seed)
        dataset = dataset.map(lambda x, y: (tf.clip_by_value(augmentation(x, training=True), 0, 255), y),
                              num_parallel_calls=autotune)
    return dataset.prefetch(autotune)

In [ ]:
train_files, train_labels = list_split(PROCESSED_DIR, "train")
val_files, val_labels = list_split(PROCESSED_DIR, "val")
test_files, test_labels = list_split(PROCESSED_DIR, "test")
train_ds = make_dataset(train_files, train_labels, IMAGE_SIZE, BATCH_SIZE, training=True, seed=SEED)
val_ds = make_dataset(val_files, val_labels, IMAGE_SIZE, BATCH_SIZE, training=False)

images, labels = next(iter(train_ds))
print("train / val / test images:", len(train_files), len(val_files), len(test_files))
print("batch images:", images.shape, images.dtype, "| batch labels:", labels.shape, labels.dtype,
      "| label range:", int(tf.reduce_min(labels)), "-", int(tf.reduce_max(labels)))

overlap = manifest[manifest.split != "test"].groupby("source_image")["split"].nunique()
print("Source photos appearing in both train and val:", int((overlap > 1).sum()))
print("Test crops come from the official test photos (separate files and naming scheme).")

## Section 11 — Baseline CNN

A deliberately small custom network (`build_baseline_cnn`, below):

* **Rescaling** — pixels to [0, 1].
* **4 convolution blocks** (32 → 64 → 128 → 256 filters). Each block is Conv 3×3 → BatchNorm → ReLU → MaxPool.
  Early blocks learn edges and colours; later blocks learn shell outlines and internal texture.
  Each max-pool halves the resolution.
* **Global average pooling** — one number per filter. It has far fewer parameters than `Flatten`, which limits overfitting.
* **Dropout 0.3** — randomly drops features during training (regularisation).
* **Dense softmax (11)** — one probability per class.

In [ ]:
# --- copied from src/model.py ---
layers = tf.keras.layers


BACKBONE_LAYER_NAME = "mobilenetv2_backbone"


def build_baseline_cnn(image_size: int, num_classes: int, dropout: float = 0.3) -> tf.keras.Model:
    """Small custom CNN: 4 x (Conv -> ReLU -> MaxPool), global average pooling, softmax.

    Filters double each block (32 -> 256) while the image shrinks, so the network
    learns simple edges first and more complex shell/texture patterns later.
    Batch normalisation after each convolution keeps training stable at a
    normal learning rate. Global average pooling (instead of Flatten) keeps the
    parameter count small, which limits overfitting on ~9k training images.
    """
    inputs = layers.Input(shape=(image_size, image_size, 3), name="image")
    x = layers.Rescaling(1.0 / 255, name="rescale_0_1")(inputs)
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    return tf.keras.Model(inputs, outputs, name="ovascope_baseline_cnn")


def count_parameters(model: tf.keras.Model) -> dict:
    trainable = sum(int(tf.size(w)) for w in model.trainable_weights)
    total = sum(int(tf.size(w)) for w in model.weights)
    return {"total": total, "trainable": trainable, "non_trainable": total - trainable}

In [ ]:
baseline_preview = build_baseline_cnn(IMAGE_SIZE, len(CLASS_NAMES))
baseline_preview.summary()
print(count_parameters(baseline_preview))

## Section 12 — Baseline training

* **Optimiser:** Adam, learning rate 1e-3. It adapts per-parameter step sizes and is a robust default.
* **Loss:** sparse categorical cross-entropy (integer labels, softmax outputs).
* **Epochs:** up to 30, with these callbacks:
  * **EarlyStopping** (patience 5, on validation loss), which **restores the best weights**
  * **ReduceLROnPlateau**, which halves the learning rate after 2 stagnant epochs
  * **ModelCheckpoint**, which keeps the best epoch
  * **BackupAndRestore**, which lets an interrupted run resume
  * **EpochMetrics**, which measures the seven evaluation metrics on the training and validation sets
    after every epoch (see 12.3)
* **No class weights:** Section 7 shows the classes are balanced.

The code that does this is shown in 12.1–12.4; `train_baseline` runs it and saves the result as a new
model version. The log at the end of this section is the actual training output.

### 12.1 Model versioning (registry)

Each trained model gets its own folder `models/versions/vNNN_<kind>/`. Activating a version only rewrites
the small pointer file `active_model.json` (atomically), so a previous version can always be restored.

In [ ]:
# --- copied from src/registry.py ---
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def versions_dir(models_dir: Path) -> Path:
    return Path(models_dir) / "versions"


def list_versions(models_dir: Path) -> list[dict]:
    out = []
    folder = versions_dir(models_dir)
    if not folder.exists():
        return out
    for path in sorted(folder.iterdir()):
        meta = path / "model_metadata.json"
        if meta.exists():
            out.append(json.loads(meta.read_text()))
    return out


def next_version_name(models_dir: Path, kind: str) -> str:
    numbers = [int(m.group(1)) for p in versions_dir(models_dir).glob("v*")
               if (m := re.match(r"v(\d+)_", p.name))]
    return f"v{(max(numbers) if numbers else 0) + 1:03d}_{kind}"


def get_active_version(models_dir: Path) -> str | None:
    pointer = Path(models_dir) / "active_model.json"
    if not pointer.exists():
        return None
    return json.loads(pointer.read_text()).get("version")


def version_path(models_dir: Path, version: str) -> Path:
    path = versions_dir(models_dir) / version
    if not (path / "model.keras").exists():
        raise FileNotFoundError(f"Model version '{version}' not found in {versions_dir(models_dir)}")
    return path


def activate_version(models_dir: Path, version: str) -> dict:
    """Point the application at `version` (atomic pointer write + stable-name copies)."""
    models_dir = Path(models_dir)
    source = version_path(models_dir, version)
    current = get_active_version(models_dir)
    previous = current
    if current == version:   # re-activating the same version keeps the existing rollback target
        previous = json.loads((models_dir / "active_model.json").read_text()).get("previous")
    for name, target in [("model.keras", "ovascope_best_model.keras"), ("model.h5", "ovascope_best_model.h5"),
                         ("class_mapping.json", "class_mapping.json")]:
        if (source / name).exists():
            tmp = models_dir / f".{target}.tmp"
            shutil.copy2(source / name, tmp)
            os.replace(tmp, models_dir / target)
    pointer = {"version": version, "previous": previous, "activated_at": utc_now()}
    tmp = models_dir / ".active_model.json.tmp"
    tmp.write_text(json.dumps(pointer, indent=2))
    os.replace(tmp, models_dir / "active_model.json")   # atomic: readers never see a half-written file
    return pointer


def rollback(models_dir: Path) -> dict:
    pointer = json.loads((Path(models_dir) / "active_model.json").read_text())
    if not pointer.get("previous"):
        raise ValueError("No previous model version to roll back to")
    return activate_version(models_dir, pointer["previous"])

### 12.2 Evaluation metrics

Seven metrics are reported for every model, every epoch and every class. With $TP_k$, $FP_k$, $FN_k$ the true
positives, false positives and false negatives of class $k$, and $p_{i,k}$ the predicted probability of class $k$
for image $i$:

| Metric | Definition | What it tells us |
|---|---|---|
| **Accuracy** | correct predictions / all predictions | overall share of eggs classified correctly. Per class it is the share of that class's eggs classified correctly (equal to recall) |
| **Precision** | $TP_k / (TP_k + FP_k)$ | when the model says "class *k*", how often it is right |
| **Recall (sensitivity)** | $TP_k / (TP_k + FN_k)$ | how many eggs of class *k* the model finds; a missed parasite is a false negative |
| **F1-score** | $2 \cdot P \cdot R / (P + R)$ | balance of precision and recall; the main model-selection metric |
| **ROC-AUC** | area under the ROC curve (true- vs false-positive rate over all thresholds), one class vs the rest | how well the probability ranks eggs of class *k* above other eggs, independent of a threshold. 0.5 = chance, 1 = perfect |
| **PR-AUC** | average precision: area under the precision–recall curve, one class vs the rest | like ROC-AUC, but focused on the positive class; stricter when a class is only 1/11 of the data. Chance level ≈ 0.09 |
| **Log loss** | $-\frac{1}{N}\sum_i \log p_{i,y_i}$ (categorical cross-entropy) | penalises confident wrong answers; measures how trustworthy the probabilities are. Lower is better; chance level = ln 11 ≈ 2.40 |

Precision, recall, F1, ROC-AUC and PR-AUC are **macro averages**: computed per class, then averaged, so every
parasite class counts equally. Balanced accuracy (mean per-class recall) and weighted averages are kept too.

In [ ]:
# --- copied from src/evaluation.py (metrics) ---
HEADLINE_METRICS = {"accuracy": "Accuracy", "macro_precision": "Precision", "macro_recall": "Recall",
                    "macro_f1": "F1-score", "roc_auc_macro": "ROC-AUC", "pr_auc_macro": "PR-AUC",
                    "log_loss": "Log loss"}


def predict_dataset(model, dataset):
    """Run the model over a (non-shuffled) tf.data dataset. Returns (y_true, probabilities)."""
    y_true, probabilities = [], []
    for images, labels in dataset:
        probabilities.append(model(images, training=False).numpy())
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(probabilities)


def one_vs_rest_auc(y_true, probabilities, num_classes: int):
    """Per-class ROC-AUC and PR-AUC (average precision), one class against all others.
    A class with no positive (or no negative) samples gets None: its AUC is undefined."""
    roc, pr = [], []
    for k in range(num_classes):
        positives = (np.asarray(y_true) == k)
        if 0 < positives.sum() < len(positives):
            roc.append(float(roc_auc_score(positives, probabilities[:, k])))
            pr.append(float(average_precision_score(positives, probabilities[:, k])))
        else:
            roc.append(None)
            pr.append(None)
    return roc, pr


def _mean_defined(values):
    defined = [v for v in values if v is not None]
    return float(np.mean(defined)) if defined else None


def compute_metrics(y_true, probabilities, class_names: list[str]) -> dict:
    y_true = np.asarray(y_true)
    y_pred = probabilities.argmax(axis=1)
    eps = 1e-7
    true_class_probability = np.clip(probabilities[np.arange(len(y_true)), y_true], eps, 1.0)
    sample_log_loss = -np.log(true_class_probability)
    loss = float(sample_log_loss.mean())
    labels = list(range(len(class_names)))
    macro = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    weighted = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    per_class = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    roc, pr = one_vs_rest_auc(y_true, probabilities, len(class_names))
    return {
        "samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "loss": loss,
        "log_loss": loss,
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro[0]), "macro_recall": float(macro[1]), "macro_f1": float(macro[2]),
        "weighted_precision": float(weighted[0]), "weighted_recall": float(weighted[1]),
        "weighted_f1": float(weighted[2]),
        "roc_auc_macro": _mean_defined(roc),
        "pr_auc_macro": _mean_defined(pr),
        "per_class": {name: {
            # class-wise accuracy = share of this class's samples classified correctly (equals recall)
            "accuracy": float(per_class[1][i]),
            "precision": float(per_class[0][i]), "recall": float(per_class[1][i]),
            "f1": float(per_class[2][i]), "roc_auc": roc[i], "pr_auc": pr[i],
            "log_loss": float(sample_log_loss[y_true == i].mean()) if (y_true == i).any() else None,
            "support": int(per_class[3][i])}
            for i, name in enumerate(class_names)},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def headline_row(metrics: dict, digits: int = 4) -> dict:
    """The seven headline metrics (plus balanced accuracy) of one evaluation, as a flat row."""
    keys = list(HEADLINE_METRICS) + ["balanced_accuracy"]
    return {HEADLINE_METRICS.get(k, k.replace("_", " ")): None if metrics.get(k) is None else round(metrics[k], digits)
            for k in keys}


def per_class_frame(metrics: dict) -> pd.DataFrame:
    """Per-class table: accuracy, precision, recall, F1, ROC-AUC, PR-AUC, log loss, support."""
    frame = pd.DataFrame(metrics["per_class"]).T
    frame = frame[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "log_loss", "support"]]
    frame.columns = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC", "Log loss", "Support"]
    frame["Support"] = frame["Support"].astype(int)
    return frame

### 12.3 Training loop and per-epoch metrics

Keras reports `accuracy` and `loss` as running averages over the **augmented** training batches of each
epoch. That alone cannot show precision, recall, F1, ROC-AUC, PR-AUC or log loss. So the `EpochMetrics`
callback predicts the **full training set (without augmentation)** and the **validation set** after every
epoch and computes all seven metrics with `compute_metrics`. The values go into the epoch logs, so they
are printed during training and saved in `training_history.csv` as `train_<metric>` and `val_<metric>`
columns. The callback only measures; it does not change training, and the test set is never touched.

In [ ]:
# --- copied from src/train.py (data, callbacks and the training loop) ---
def class_names_from(cfg: dict) -> list[str]:
    return [c["display_name"] for c in load_class_map(cfg["paths"]["processed_dir"])]


def load_train_val(cfg: dict):
    """tf.data datasets for training (augmented, shuffled) and validation (plain)."""
    t = cfg["training"]
    processed = cfg["paths"]["processed_dir"]
    train_files, train_labels = list_split(processed, "train")
    val_files, val_labels = list_split(processed, "val")
    if not train_files or not val_files:
        raise FileNotFoundError(f"No training/validation crops found in {processed}. Run scripts/prepare_data.py")
    seed = cfg["project"]["random_seed"]
    train_ds = make_dataset(train_files, train_labels, t["image_size"], t["batch_size"], training=True, seed=seed)
    val_ds = make_dataset(val_files, val_labels, t["image_size"], t["batch_size"], training=False)
    return train_ds, val_ds


class EpochMetrics(tf.keras.callbacks.Callback):
    """At the end of every epoch, measure the seven headline metrics on the full training set
    (without augmentation) and on the validation set, and add them to the epoch logs.

    Columns added: train_/val_ + accuracy, precision, recall, f1, roc_auc, pr_auc, log_loss
    (precision/recall/F1 are macro averages; ROC-AUC and PR-AUC are one-vs-rest macro averages).
    They are written to the CSV log and the training history together with Keras' own
    `accuracy`/`loss` (running averages over the *augmented* batches) and `val_accuracy`/`val_loss`.
    Must be placed before CSVLogger/EarlyStopping in the callback list so they see the new values.
    """

    KEYS = {"accuracy": "accuracy", "precision": "macro_precision", "recall": "macro_recall", "f1": "macro_f1",
            "roc_auc": "roc_auc_macro", "pr_auc": "pr_auc_macro", "log_loss": "log_loss"}

    def __init__(self, splits: dict, class_names: list[str], image_size: int, batch_size: int):
        """splits: {"train": (file_paths, labels), "val": (file_paths, labels)}"""
        super().__init__()
        self.datasets = {name: (make_dataset(files, labels, image_size, batch_size, training=False)
                                .map(lambda x, y: x), np.asarray(labels))
                         for name, (files, labels) in splits.items()}
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        parts = []
        for name, (images, labels) in self.datasets.items():
            probabilities = self.model.predict(images, verbose=0)
            metrics = compute_metrics(labels, probabilities, self.class_names)
            for short, key in self.KEYS.items():
                logs[f"{name}_{short}"] = float("nan") if metrics[key] is None else metrics[key]
            parts.append(f"{name}: f1={metrics['macro_f1']:.4f} roc_auc={metrics['roc_auc_macro'] or float('nan'):.4f} "
                         f"pr_auc={metrics['pr_auc_macro'] or float('nan'):.4f} log_loss={metrics['log_loss']:.4f}")
        print(f"  epoch {epoch + 1} metrics | " + " | ".join(parts))


def compile_model(model, learning_rate: float) -> None:
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])


def fit(model, train_ds, val_ds, epochs: int, learning_rate: float, run_dir: Path, cfg: dict,
        phase: str = "train", epoch_metrics: EpochMetrics | None = None) -> tuple[pd.DataFrame, float]:
    """Compile + fit with the standard callbacks. Returns (history, seconds).

    * EpochMetrics (optional) adds accuracy, precision, recall, F1, ROC-AUC, PR-AUC and log loss
      on the training and validation sets to every epoch's history.
    * ModelCheckpoint keeps the epoch with the lowest validation loss.
    * EarlyStopping stops after `patience` epochs without improvement and restores the best weights.
    * ReduceLROnPlateau halves the learning rate when validation loss stalls.
    * BackupAndRestore makes an interrupted run resumable from its last epoch.
    """
    t = cfg["training"]
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    compile_model(model, learning_rate)
    callbacks = [epoch_metrics] if epoch_metrics is not None else []
    callbacks += [
        tf.keras.callbacks.ModelCheckpoint(str(run_dir / f"best_{phase}.keras"), monitor="val_loss",
                                           save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=t["early_stopping_patience"],
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=t["reduce_lr_patience"],
                                             min_lr=1e-7),
        tf.keras.callbacks.CSVLogger(str(run_dir / f"history_{phase}.csv")),
        tf.keras.callbacks.BackupAndRestore(str(run_dir / f"backup_{phase}")),
    ]
    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    seconds = time.time() - start
    frame = pd.DataFrame(history.history)
    frame.insert(0, "epoch", np.arange(1, len(frame) + 1))
    if "train_log_loss" in frame:   # put the per-epoch metric columns in a readable order
        ordered = ["epoch", "loss", "accuracy"] + [f"{s}_{m}" for s in ("train", "val") for m in EpochMetrics.KEYS]
        ordered = [c for c in ordered if c in frame] + [c for c in frame if c not in ordered]
        frame = frame[ordered]
    frame["phase"] = phase
    if "learning_rate" not in frame:
        frame["learning_rate"] = float(model.optimizer.learning_rate.numpy())
    return frame, seconds


def validation_metrics(model, val_ds, class_names) -> dict:
    y_true, probabilities = predict_dataset(model, val_ds)
    return compute_metrics(y_true, probabilities, class_names)

### 12.4 Saving a model version

`save_version` writes the native `.keras` model and an `.h5` export. It reloads both files and checks that
they predict the same values (`export_and_verify_h5`). It also writes the class mapping, the training
history (including the per-epoch metrics), the configuration, the model summary and metadata with
SHA-256 checksums.

In [ ]:
# --- copied from src/train.py (saving) ---
def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def export_and_verify_h5(model, keras_path: Path, h5_path: Path, image_size: int) -> dict:
    """Save a legacy .h5 copy, reload both files, and confirm they predict the same thing."""
    model.save(h5_path)
    sample = np.random.default_rng(0).uniform(0, 255, (4, image_size, image_size, 3)).astype("float32")
    from_keras = tf.keras.models.load_model(keras_path, compile=False).predict(sample, verbose=0)
    from_h5 = tf.keras.models.load_model(h5_path, compile=False).predict(sample, verbose=0)
    difference = float(np.abs(from_keras - from_h5).max())
    if difference > 1e-4:
        raise RuntimeError(f".h5 export differs from .keras model (max abs diff {difference})")
    return {"h5_reload_ok": True, "max_abs_difference_vs_keras": difference}


def save_version(model, kind: str, cfg: dict, history: pd.DataFrame, val_metrics: dict, extra: dict,
                 parent_version: str | None = None) -> str:
    """Write a new models/versions/<version>/ folder and return the version name."""
    models_dir = cfg["paths"]["models_dir"]
    version = next_version_name(models_dir, kind)
    out = versions_dir(models_dir) / version
    out.mkdir(parents=True)
    image_size = cfg["training"]["image_size"]

    keras_path, h5_path = out / "model.keras", out / "model.h5"
    model.save(keras_path)
    h5_check = export_and_verify_h5(model, keras_path, h5_path, image_size)
    class_map = load_class_map(cfg["paths"]["processed_dir"])
    (out / "class_mapping.json").write_text(json.dumps(class_map, indent=2))
    history.to_csv(out / "training_history.csv", index=False)
    training_config = {"training": cfg["training"], "retraining": cfg["retraining"],
                       "dataset": {k: v for k, v in cfg["dataset"].items() if k != "archives"},
                       "random_seed": cfg["project"]["random_seed"]}
    (out / "training_config.json").write_text(json.dumps(training_config, indent=2))
    summary_lines = []
    model.summary(print_fn=lambda line, *a, **k: summary_lines.append(line), expand_nested=False)
    (out / "model_summary.txt").write_text("\n".join(summary_lines))

    best_epoch = int(history["val_loss"].idxmin()) + 1
    metadata = {
        "version": version, "kind": kind, "architecture": model.name,
        "parent_version": parent_version,
        "input_shape": [image_size, image_size, 3], "input_range": "[0, 255] RGB (model rescales internally)",
        "num_classes": len(class_map),
        "created_at": utc_now(),
        "epochs_completed": int(len(history)), "best_epoch": best_epoch,
        "final_learning_rate": float(history["learning_rate"].iloc[-1]),
        "final_train_loss": float(history["loss"].iloc[-1]), "final_train_accuracy": float(history["accuracy"].iloc[-1]),
        "best_val_loss": float(history["val_loss"].min()),
        "validation_metrics": {k: v for k, v in val_metrics.items() if k not in ("confusion_matrix", "per_class")},
        "parameters": count_parameters(model),
        "files": {"keras": "model.keras", "h5": "model.h5"},
        "model_size_mb": round(keras_path.stat().st_size / 1e6, 2),
        "sha256": {"model.keras": sha256_of(keras_path), "model.h5": sha256_of(h5_path)},
        "h5_export": h5_check,
        "environment": describe_environment(),
        **extra,
    }
    (out / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
    return version


def update_metadata(cfg: dict, version: str, **fields) -> dict:
    path = versions_dir(cfg["paths"]["models_dir"]) / version / "model_metadata.json"
    metadata = json.loads(path.read_text())
    metadata.update(fields)
    path.write_text(json.dumps(metadata, indent=2))
    return metadata

### 12.5 Baseline training recipe and run

In [ ]:
# --- copied from src/train.py (training recipe) ---
def make_epoch_metrics(cfg: dict, class_names) -> EpochMetrics | None:
    """The per-epoch metrics callback (full training set without augmentation + validation set),
    unless disabled with training.epoch_metrics: false."""
    t = cfg["training"]
    if not t.get("epoch_metrics", True):
        return None
    processed = cfg["paths"]["processed_dir"]
    splits = {"train": list_split(processed, "train"), "val": list_split(processed, "val")}
    return EpochMetrics(splits, class_names, t["image_size"], t["batch_size"])


def train_baseline(cfg: dict, epochs: int | None = None) -> str:
    t = cfg["training"]
    set_seeds(cfg["project"]["random_seed"])
    class_names = class_names_from(cfg)
    train_ds, val_ds = load_train_val(cfg)
    tracker = make_epoch_metrics(cfg, class_names)
    model = build_baseline_cnn(t["image_size"], len(class_names))
    run_dir = cfg["paths"]["models_dir"] / "checkpoints" / "baseline_cnn"
    history, seconds = fit(model, train_ds, val_ds, epochs or t["baseline"]["epochs"],
                           t["baseline"]["learning_rate"], run_dir, cfg, phase="baseline", epoch_metrics=tracker)
    val_metrics = validation_metrics(model, val_ds, class_names)
    print(f"Baseline CNN validation: accuracy={val_metrics['accuracy']:.4f} macro_f1={val_metrics['macro_f1']:.4f}")
    return save_version(model, "baseline_cnn", cfg, history, val_metrics,
                        {"training_seconds": round(seconds, 1), "pretrained_weights": None})

In [ ]:
baseline_version = train_baseline(cfg)
baseline_version

## Section 13 — Baseline evaluation (validation set)

We report the seven metrics from 12.2 (**accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC, log loss**),
first **per epoch** (training vs validation), then **overall** and **per class** on the final model. The
validation set is used here; the test set is still untouched.

The plotting and reporting code comes first.

In [ ]:
# --- copied from src/evaluation.py (reports and plots) ---
def classification_report_frame(y_true, probabilities, class_names) -> pd.DataFrame:
    report = classification_report(y_true, probabilities.argmax(axis=1), labels=list(range(len(class_names))),
                                   target_names=class_names, output_dict=True, zero_division=0)
    return pd.DataFrame(report).transpose()


def top_confusions(metrics: dict, class_names: list[str], n: int = 5) -> list[dict]:
    matrix = np.array(metrics["confusion_matrix"])
    np.fill_diagonal(matrix, 0)
    pairs = [(matrix[i, j], i, j) for i in range(len(matrix)) for j in range(len(matrix)) if matrix[i, j] > 0]
    return [{"true": class_names[i], "predicted": class_names[j], "count": int(c)}
            for c, i, j in sorted(pairs, reverse=True)[:n]]


def plot_confusion_matrix(matrix, class_names, path: Path, title: str = "Confusion matrix"):
    matrix = np.array(matrix)
    normalised = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(10, 8.5))
    image = ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if matrix[i, j]:
                ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=8,
                        color="white" if normalised[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{title} (counts; colour = row-normalised recall)")
    fig.colorbar(image, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def plot_history(history: pd.DataFrame, path_accuracy: Path, path_loss: Path, title: str = ""):
    """Training vs validation curves. `history` needs columns accuracy, val_accuracy, loss, val_loss
    and optionally `phase` (e.g. head / fine_tune) to mark where fine-tuning starts."""
    epochs = np.arange(1, len(history) + 1)
    for metric, path in (("accuracy", path_accuracy), ("loss", path_loss)):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot(epochs, history[metric], marker="o", label=f"train {metric}")
        ax.plot(epochs, history[f"val_{metric}"], marker="o", label=f"validation {metric}")
        if "phase" in history and history["phase"].nunique() > 1:
            start = int(np.argmax(history["phase"].values != history["phase"].values[0])) + 1
            ax.axvline(start - 0.5, color="grey", linestyle="--", label="fine-tuning starts")
        ax.set_xlabel("epoch")
        ax.set_ylabel(metric)
        ax.set_title(f"{title} {metric}".strip())
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(path, dpi=130)
        plt.close(fig)


EPOCH_METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "log_loss"]


def plot_epoch_metrics(history: pd.DataFrame, path: Path, title: str = ""):
    """One panel per headline metric: training set vs validation set, epoch by epoch.

    Uses the columns written by src.train.EpochMetrics (train_<metric>, val_<metric>). The training
    values are measured on the full, un-augmented training set at the end of each epoch."""
    epochs = np.arange(1, len(history) + 1)
    names = {"accuracy": "Accuracy", "precision": "Precision (macro)", "recall": "Recall (macro)",
             "f1": "F1-score (macro)", "roc_auc": "ROC-AUC (macro, OvR)", "pr_auc": "PR-AUC (macro, OvR)",
             "log_loss": "Log loss (cross-entropy)"}
    fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
    start = None
    if "phase" in history and history["phase"].nunique() > 1:
        start = int(np.argmax(history["phase"].values != history["phase"].values[0])) + 1
    for ax, metric in zip(axes.flat, EPOCH_METRICS):
        for split, style in (("train", "o-"), ("val", "s-")):
            column = f"{split}_{metric}"
            if column in history:
                ax.plot(epochs, history[column], style, markersize=3.5,
                        label="training set" if split == "train" else "validation set")
        if start:
            ax.axvline(start - 0.5, color="grey", linestyle="--", label="fine-tuning starts")
        ax.set_title(names[metric])
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    axes.flat[0].legend()
    last = axes.flat[-1]
    if "learning_rate" in history:
        last.plot(epochs, history["learning_rate"], "k.-")
        last.set_yscale("log")
        last.set_title("Learning rate")
        last.set_xlabel("epoch")
        last.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        last.grid(alpha=0.3)
    else:
        last.axis("off")
    fig.suptitle(f"{title} - metrics per epoch".strip(" -"), fontsize=14)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_roc_pr_curves(y_true, probabilities, class_names, path: Path, title: str = ""):
    """One-vs-rest ROC and precision-recall curves for every class, with the per-class AUCs."""
    y_true = np.asarray(y_true)
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(17, 7))
    colours = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
    for k, (name, colour) in enumerate(zip(class_names, colours)):
        positives = y_true == k
        if not 0 < positives.sum() < len(positives):
            continue
        fpr, tpr, _ = roc_curve(positives, probabilities[:, k])
        precision, recall, _ = precision_recall_curve(positives, probabilities[:, k])
        ax_roc.plot(fpr, tpr, color=colour, lw=1.4,
                    label=f"{name} ({roc_auc_score(positives, probabilities[:, k]):.3f})")
        ax_pr.plot(recall, precision, color=colour, lw=1.4,
                   label=f"{name} ({average_precision_score(positives, probabilities[:, k]):.3f})")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance")
    ax_roc.set(xlabel="false positive rate", ylabel="true positive rate (recall)", title="ROC curves (one-vs-rest)")
    ax_pr.set(xlabel="recall", ylabel="precision", title="Precision-recall curves (one-vs-rest)",
              xlim=(0, 1.01), ylim=(0, 1.02))
    for ax in (ax_roc, ax_pr):
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7.5, loc="lower right" if ax is ax_roc else "lower left", title="class (AUC)")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_prediction_examples(file_paths, y_true, probabilities, class_names, path: Path, per_group: int = 6,
                             seed: int = 0):
    """Three rows: correct, incorrect, lowest-confidence predictions (real examples, not curated)."""
    from PIL import Image
    rng = np.random.default_rng(seed)
    y_pred = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    correct = np.nonzero(y_pred == y_true)[0]
    wrong = np.nonzero(y_pred != y_true)[0]
    groups = [
        ("Correct", rng.choice(correct, min(per_group, len(correct)), replace=False) if len(correct) else []),
        ("Incorrect", rng.choice(wrong, min(per_group, len(wrong)), replace=False) if len(wrong) else []),
        ("Lowest confidence", np.argsort(confidence)[:per_group]),
    ]
    fig, axes = plt.subplots(3, per_group, figsize=(2.6 * per_group, 9.2))
    for row, (label, indices) in enumerate(groups):
        for col in range(per_group):
            ax = axes[row, col]
            ax.axis("off")
            if col >= len(indices):
                continue
            i = indices[col]
            ax.imshow(Image.open(file_paths[i]).convert("RGB"))
            ok = y_pred[i] == y_true[i]
            ax.set_title(f"true: {class_names[y_true[i]][:20]}\npred: {class_names[y_pred[i]][:20]}\n"
                         f"conf: {confidence[i]:.2f}", fontsize=7.5, color="green" if ok else "red")
        axes[row, 0].text(-0.15, 0.5, label, transform=axes[row, 0].transAxes, rotation=90,
                          va="center", ha="right", fontsize=11, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def measure_inference_time(model, image_size: int, runs: int = 50) -> dict:
    """Median single-image latency on the current device, using a compiled forward pass
    (tf.function) exactly as src.prediction.Predictor does in the API."""
    import tensorflow as tf
    batch = tf.random.uniform((1, image_size, image_size, 3), 0, 255)
    forward = tf.function(lambda x: model(x, training=False))
    for _ in range(5):
        forward(batch).numpy()
    timings = []
    for _ in range(runs):
        start = time.perf_counter()
        forward(batch).numpy()
        timings.append((time.perf_counter() - start) * 1000)
    return {"median_ms": float(np.median(timings)), "p95_ms": float(np.percentile(timings, 95)),
            "runs": runs, "device": "GPU" if tf.config.list_physical_devices("GPU") else "CPU"}


def save_evaluation(metrics: dict, report: pd.DataFrame, class_names, out_dir: Path, prefix: str) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{prefix}_metrics.json").write_text(json.dumps(metrics, indent=2))
    report.to_csv(out_dir / f"{prefix}_classification_report.csv")


def evaluate_model(model, file_paths, labels, class_names, image_size, batch_size, out_dir: Path,
                   prefix: str, plot_examples: bool = True) -> dict:
    """Full evaluation of one model on one split: metrics JSON, report CSV, confusion matrix,
    ROC / precision-recall curves and prediction examples."""
    dataset = make_dataset(file_paths, labels, image_size, batch_size, training=False)
    y_true, probabilities = predict_dataset(model, dataset)
    metrics = compute_metrics(y_true, probabilities, class_names)
    metrics["top_confusions"] = top_confusions(metrics, class_names)
    report = classification_report_frame(y_true, probabilities, class_names)
    save_evaluation(metrics, report, class_names, out_dir, prefix)
    plot_confusion_matrix(metrics["confusion_matrix"], class_names, Path(out_dir) / f"{prefix}_confusion_matrix.png",
                          title=prefix.replace("_", " "))
    plot_roc_pr_curves(y_true, probabilities, class_names, Path(out_dir) / f"{prefix}_roc_pr_curves.png",
                       title=prefix.replace("_", " "))
    if plot_examples:
        plot_prediction_examples(file_paths, y_true, probabilities, class_names,
                                 Path(out_dir) / f"{prefix}_predictions.png")
    np.save(Path(out_dir) / f"{prefix}_probabilities.npy", probabilities)
    return metrics

In [ ]:
def load_version(version):
    folder = version_path(cfg["paths"]["models_dir"], version)
    return tf.keras.models.load_model(folder / "model.keras", compile=False), folder


summary_row = headline_row     # one row of the seven headline metrics (+ balanced accuracy)
METRIC_LABELS = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC", "Log loss"]


def epoch_table(history):
    """Per-epoch metrics on the training set (no augmentation) and the validation set."""
    columns = {}
    for split, label in (("train", "Training set"), ("val", "Validation set")):
        for metric, name in zip(EPOCH_METRICS, METRIC_LABELS):
            columns[(label, name)] = history[f"{split}_{metric}"].values
    table = pd.DataFrame(columns, index=pd.Index(history["epoch"], name="epoch"))
    if "phase" in history:
        table.insert(0, ("", "phase"), history["phase"].values)
    table[("", "learning rate")] = history["learning_rate"].values
    return table.round(4)


def show_images(paths, figsize):
    fig, axes = plt.subplots(1, len(paths), figsize=figsize)
    for ax, path in zip(np.atleast_1d(axes), paths):
        ax.imshow(Image.open(path))
        ax.axis("off")
    plt.tight_layout()
    plt.show()


baseline_model, baseline_folder = load_version(baseline_version)
baseline_history = pd.read_csv(baseline_folder / "training_history.csv")
plot_history(baseline_history, PLOTS / f"{baseline_version}_accuracy_curve.png",
             PLOTS / f"{baseline_version}_loss_curve.png", title="Baseline CNN")
plot_epoch_metrics(baseline_history, PLOTS / f"{baseline_version}_epoch_metrics.png", title="Baseline CNN")
baseline_val = evaluate_model(baseline_model, val_files, val_labels, CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE,
                              METRICS, f"{baseline_version}_val", plot_examples=False)
print("Baseline CNN, best epoch (lowest validation loss):", int(baseline_history["val_loss"].idxmin()) + 1,
      "of", len(baseline_history))

### 13.1 Metrics per epoch

Circles are the **training set** (measured without augmentation at the end of each epoch); squares are the
**validation set**. The last panel shows the learning rate, which `ReduceLROnPlateau` lowers.

In [ ]:
show_images([PLOTS / f"{baseline_version}_epoch_metrics.png"], (20, 8.5))
epoch_table(baseline_history)

**Interpretation.** _(filled in after execution)_

### 13.2 Keras learning curves and validation confusion matrix

In [ ]:
show_images([PLOTS / f"{baseline_version}_accuracy_curve.png", PLOTS / f"{baseline_version}_loss_curve.png",
             METRICS / f"{baseline_version}_val_confusion_matrix.png"], (20, 5.5))

### 13.3 Overall validation performance

In [ ]:
pd.Series(headline_row(baseline_val), name="Baseline CNN, validation").to_frame()

### 13.4 Validation performance of each class

In [ ]:
show_images([METRICS / f"{baseline_version}_val_roc_pr_curves.png"], (17, 7))
per_class_frame(baseline_val).round(3)

**Interpretation.** The baseline CNN reaches **0.861 validation accuracy and 0.860 macro-F1** after 30 epochs.
* **Learning dynamics:** validation loss started high (3.19 after epoch 1) and was unstable for the first
  epochs while BatchNorm statistics settled at learning rate 1e-3. ReduceLROnPlateau lowered the rate
  step by step, and validation loss then fell steadily to 0.431.
* **Not overfitting:** final training accuracy (0.822, measured *with* augmentation and dropout) is
  *below* validation accuracy (0.861), so the network is not memorising the training set.
* **Possibly under-trained:** the best epoch was the last one (30) and early stopping never triggered.
  The baseline was still improving when the 30-epoch budget ran out.
* **Run-to-run variation:** three complete runs of this notebook gave baseline validation macro-F1 of
  0.843, 0.890 and 0.860. Seeds are fixed, but GPU kernels are not fully deterministic, and a small CNN
  trained from scratch is sensitive to that. Differences of a few points between baseline runs are noise.
* **Per class:** F1 is lowest for *Capillaria* (0.78), *Ascaris* (0.79) and *Paragonimus* (0.83), and
  highest for *Fasciolopsis* (0.93), *Taenia* (0.91) and *Hookworm* (0.91).

## Section 14 — Improved model: MobileNetV2 transfer learning

**Why MobileNetV2?** It has about 3.5 M parameters (around 14 MB), is fast on a CPU (so cheap to deploy), and its
ImageNet features (edges, textures, shapes) transfer well to microscopy. EfficientNetB0 is a little more
accurate on ImageNet, but slower on CPU, and Keras's version hides its own input rescaling inside the
model, which makes double normalisation an easy mistake. We train only this one backbone rather than
trying several large ones.

* **Pretrained weights:** ImageNet (downloaded once by Keras and cached). If a platform has no internet,
  `build_transfer_model(weights=<local .h5 path>)` (below) uses a local copy. The cell fails loudly rather than
  silently using random weights.
* **Custom head:** global average pooling → dropout 0.3 → dense softmax (11).
* **Stage 1 (head):** backbone **frozen**, so only the head learns (lr 1e-3).
* **Stage 2 (fine-tuning):** layers ≥ 100 of 154 unfrozen, lr **1e-5**. The upper layers hold
  ImageNet-specific shapes worth adapting; the lower layers hold generic edges, so they stay frozen.
  **BatchNorm layers stay frozen** and the backbone runs in inference mode. Updating BN statistics
  with small batches would erase the pretrained features.

In [ ]:
# --- copied from src/model.py (transfer learning) ---
def build_transfer_model(image_size: int, num_classes: int, dropout: float = 0.3,
                         weights: str | None = "imagenet") -> tf.keras.Model:
    """MobileNetV2 backbone (ImageNet weights) + custom classification head.

    * Rescaling(1/127.5, offset=-1) maps [0,255] -> [-1,1], exactly what
      tf.keras.applications.mobilenet_v2.preprocess_input does.
    * The backbone is called with training=False so its BatchNormalization
      layers keep their ImageNet statistics, even after we unfreeze layers for
      fine-tuning (updating BN statistics on small batches destroys pretrained features).
    * Head: global average pooling -> dropout -> dense softmax over 11 classes.

    `weights` may be "imagenet" (downloaded once and cached in ~/.keras),
    a path to a local .h5 weights file (offline Kaggle/Colab), or None.
    """
    try:
        backbone = tf.keras.applications.MobileNetV2(
            input_shape=(image_size, image_size, 3), include_top=False, weights=weights)
    except Exception as error:
        raise RuntimeError(
            "Could not load MobileNetV2 pretrained weights "
            f"({error.__class__.__name__}: {error}). If this machine has no internet, download "
            "'mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5' elsewhere and "
            "pass its path as weights=..., or attach it as a Kaggle dataset.") from None
    backbone._name = BACKBONE_LAYER_NAME
    backbone.trainable = False

    inputs = layers.Input(shape=(image_size, image_size, 3), name="image")
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="rescale_minus1_1")(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="head_pool")(x)
    x = layers.Dropout(dropout, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    return tf.keras.Model(inputs, outputs, name="ovascope_mobilenetv2")


def get_backbone(model: tf.keras.Model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            return layer
    return None


def unfreeze_top_of_backbone(model: tf.keras.Model, from_layer: int) -> int:
    """Make backbone layers [from_layer:] trainable (BatchNorm layers stay frozen).

    Lower layers detect generic edges/colours that transfer well from ImageNet;
    the upper layers encode ImageNet-specific shapes, so those are the ones worth
    adapting to parasite-egg morphology. Returns the number of trainable layers.
    """
    backbone = get_backbone(model)
    backbone.trainable = True
    trainable = 0
    for index, layer in enumerate(backbone.layers):
        is_bn = isinstance(layer, layers.BatchNormalization)
        layer.trainable = index >= from_layer and not is_bn
        trainable += int(layer.trainable)
    return trainable

In [ ]:
transfer_preview = build_transfer_model(IMAGE_SIZE, len(CLASS_NAMES))
transfer_preview.summary()
backbone = get_backbone(transfer_preview)
print("Backbone layers:", len(backbone.layers), "| trainable now:", sum(l.trainable for l in backbone.layers))
print(count_parameters(transfer_preview))

## Section 15 — Improved model training  &  Section 16 — Fine-tuning

`train_transfer` (below) trains the head and saves the best head checkpoint. It then fine-tunes and
evaluates **both** stages on validation data, and keeps whichever has the higher validation macro-F1.
The `EpochMetrics` callback records the seven metrics after every epoch of both stages.
The log below is the actual training output.

In [ ]:
# --- copied from src/train.py (transfer-learning recipe) ---
def train_transfer(cfg: dict, head_epochs: int | None = None, fine_tune_epochs: int | None = None,
                   weights: str | None = "imagenet") -> str:
    """Stage 2+3: train the head on a frozen backbone. Stage 4: fine-tune the upper backbone,
    keep whichever checkpoint has the better validation macro-F1."""
    t, tt = cfg["training"], cfg["training"]["transfer"]
    set_seeds(cfg["project"]["random_seed"])
    class_names = class_names_from(cfg)
    train_ds, val_ds = load_train_val(cfg)
    tracker = make_epoch_metrics(cfg, class_names)
    model = build_transfer_model(t["image_size"], len(class_names), dropout=tt["dropout"], weights=weights)
    run_dir = cfg["paths"]["models_dir"] / "checkpoints" / "mobilenetv2"

    head_history, head_seconds = fit(model, train_ds, val_ds, head_epochs or tt["head_epochs"],
                                     tt["head_learning_rate"], run_dir, cfg, phase="head",
                                     epoch_metrics=tracker)
    head_metrics = validation_metrics(model, val_ds, class_names)
    head_weights = model.get_weights()
    print(f"Head-only validation: accuracy={head_metrics['accuracy']:.4f} macro_f1={head_metrics['macro_f1']:.4f}")

    fine_tune_info = {"performed": False}
    history, seconds, final_metrics = head_history, head_seconds, head_metrics
    epochs = fine_tune_epochs if fine_tune_epochs is not None else tt["fine_tune_epochs"]
    if epochs > 0:
        trainable_layers = unfreeze_top_of_backbone(model, tt["fine_tune_from_layer"])
        ft_history, ft_seconds = fit(model, train_ds, val_ds, epochs, tt["fine_tune_learning_rate"],
                                     run_dir, cfg, phase="fine_tune", epoch_metrics=tracker)
        ft_metrics = validation_metrics(model, val_ds, class_names)
        print(f"Fine-tuned validation: accuracy={ft_metrics['accuracy']:.4f} macro_f1={ft_metrics['macro_f1']:.4f}")
        kept = "fine_tune" if ft_metrics["macro_f1"] >= head_metrics["macro_f1"] else "head"
        fine_tune_info = {
            "performed": True, "unfrozen_from_layer": tt["fine_tune_from_layer"],
            "trainable_backbone_layers": trainable_layers,
            "head_val_macro_f1": head_metrics["macro_f1"], "fine_tuned_val_macro_f1": ft_metrics["macro_f1"],
            "head_val_accuracy": head_metrics["accuracy"], "fine_tuned_val_accuracy": ft_metrics["accuracy"],
            "kept": kept,
        }
        history = pd.concat([head_history, ft_history], ignore_index=True)
        history["epoch"] = np.arange(1, len(history) + 1)
        seconds = head_seconds + ft_seconds
        if kept == "head":
            model.set_weights(head_weights)
            final_metrics = head_metrics
        else:
            final_metrics = ft_metrics
    return save_version(model, "mobilenetv2", cfg, history, final_metrics,
                        {"training_seconds": round(seconds, 1), "pretrained_weights": weights or "none",
                         "fine_tuning": fine_tune_info})

In [ ]:
transfer_version = train_transfer(cfg)
transfer_model, transfer_folder = load_version(transfer_version)
transfer_meta = json.loads((transfer_folder / "model_metadata.json").read_text())
pd.Series(transfer_meta["fine_tuning"]).to_frame("value")

In [ ]:
transfer_history = pd.read_csv(transfer_folder / "training_history.csv")
plot_history(transfer_history, PLOTS / f"{transfer_version}_accuracy_curve.png",
             PLOTS / f"{transfer_version}_loss_curve.png", title="MobileNetV2")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, name in zip(axes, ["accuracy", "loss"]):
    ax.imshow(Image.open(PLOTS / f"{transfer_version}_{name}_curve.png"))
    ax.axis("off")
plt.tight_layout()
plt.show()
plot_epoch_metrics(transfer_history, PLOTS / f"{transfer_version}_epoch_metrics.png", title="MobileNetV2")

### 16.1 Metrics per epoch (head training, then fine-tuning)

The dashed line marks where fine-tuning starts. As before, circles are the training set (without
augmentation) and squares the validation set.

In [ ]:
show_images([PLOTS / f"{transfer_version}_epoch_metrics.png"], (20, 8.5))
epoch_table(transfer_history)

**Interpretation.** _(filled in after execution)_

**Interpretation.**
* **Head only (frozen backbone):** validation accuracy reached 0.81 after a *single* epoch, which shows how
  useful the ImageNet features are. It plateaued at **macro-F1 0.863**, about the same as the baseline.
* **Fine-tuning** (layers ≥ 100 unfrozen, lr 1e-5) raised validation macro-F1 to **0.920**, a gain of 5.7
  points. The fine-tuned model was therefore kept.
* **Restored weights:** the lowest validation loss (0.235) came at epoch 28 (fine-tuning epoch 13).
  EarlyStopping restored those weights at the end of training.
* **Curves:** validation loss tracks training loss closely during the head stage. During fine-tuning,
  training loss keeps falling (0.14 at the end) while validation loss flattens around 0.24–0.25, apart from
  one spike at epoch 22. This is the start of **mild overfitting**, so more epochs at this learning rate
  would bring little.

## Section 17 — Model comparison and selection (validation data only)

Both candidates are compared on the **validation** set together with model size and single-image latency.
The final model is chosen by **validation macro-F1**, with deployment cost as a tie-breaker. The test set
plays no role in this choice.

In [ ]:
# --- copied from src/train.py (model selection) ---
def select_best(cfg: dict, versions: list[str]) -> str:
    """Pick the version with the highest *validation* macro-F1 (never the test set)."""
    scores = {}
    for version in versions:
        meta = json.loads((versions_dir(cfg["paths"]["models_dir"]) / version / "model_metadata.json")
                          .read_text())
        scores[version] = meta["validation_metrics"]["macro_f1"]
    best = max(scores, key=scores.get)
    print(f"Validation macro-F1: {scores} -> selected {best}")
    return best

In [ ]:
transfer_val = evaluate_model(transfer_model, val_files, val_labels, CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE,
                              METRICS, f"{transfer_version}_val", plot_examples=False)
rows = []
for version, model, metrics in [(baseline_version, baseline_model, baseline_val),
                                (transfer_version, transfer_model, transfer_val)]:
    meta = json.loads((version_path(cfg["paths"]["models_dir"], version) / "model_metadata.json").read_text())
    with tf.device("/CPU:0"):
        cpu_latency = measure_inference_time(model, IMAGE_SIZE)["median_ms"]
    rows.append({"model": meta["architecture"], "version": version, **summary_row(metrics),
                 "parameters": meta["parameters"]["total"], "size_MB": meta["model_size_mb"],
                 "CPU_latency_ms": round(cpu_latency, 1),
                 "accel_latency_ms": round(measure_inference_time(model, IMAGE_SIZE)["median_ms"], 1),
                 "epochs": meta["epochs_completed"], "train_minutes": round(meta["training_seconds"] / 60, 1)})
comparison = pd.DataFrame(rows)
comparison.to_csv(METRICS / "model_comparison.csv", index=False)

final_version = select_best(cfg, [baseline_version, transfer_version])
activate_version(cfg["paths"]["models_dir"], final_version)
comparison

### 17.1 MobileNetV2: validation performance of each class

In [ ]:
show_images([METRICS / f"{transfer_version}_val_roc_pr_curves.png"], (17, 7))
per_class_frame(transfer_val).round(3)

**Interpretation.** _(filled in after execution)_

**Interpretation and decision.** MobileNetV2 has the higher validation macro-F1 (**0.920 vs 0.860**) and
higher accuracy, precision and recall, a margin larger than the run-to-run variation seen for the
baseline. It is 5× larger on disk (24.5 MB vs 4.8 MB). Its compiled single-image latency is still small:
**11.8 ms on CPU** vs 7.0 ms for the baseline, and 3.7 vs 1.4 ms on GPU. That is negligible next to network and
image-decoding time in the API. We therefore select **`v002_mobilenetv2`** and activate it. The decision
uses validation data only.

## Section 18 — Final held-out test evaluation

The selected model is evaluated **once** on the protected test set: 2,228 crops from the official test
photos, which were never used for training, early stopping or model selection.

In [ ]:
import shutil

final_model, final_folder = load_version(final_version)
test_metrics = evaluate_model(final_model, test_files, test_labels, CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE,
                              METRICS, "final_test")
for name in ("final_test_confusion_matrix.png", "final_test_predictions.png", "final_test_roc_pr_curves.png"):
    shutil.move(METRICS / name, PLOTS / name)
test_metrics["model_version"] = final_version
(METRICS / "final_test_metrics.json").write_text(json.dumps(test_metrics, indent=2))

update_metadata(cfg, final_version, test_metrics={k: v for k, v in test_metrics.items()
                                                  if k not in ("per_class", "confusion_matrix", "top_confusions")})
activate_version(cfg["paths"]["models_dir"], final_version)   # refresh stable-name copies
shutil.copy(final_folder / "model_metadata.json", cfg["paths"]["models_dir"] / "model_metadata.json")
shutil.copy(final_folder / "training_config.json", cfg["paths"]["models_dir"] / "training_config.json")

pd.Series(headline_row(test_metrics), name=f"{final_version}, test").to_frame()

### 18.1 Test performance of each class

In [ ]:
show_images([PLOTS / "final_test_roc_pr_curves.png"], (17, 7))
per_class_frame(test_metrics).round(3)

### 18.2 Test confusion matrix and the most frequent confusions

In [ ]:
plt.figure(figsize=(11, 9))
plt.imshow(Image.open(PLOTS / "final_test_confusion_matrix.png"))
plt.axis("off")
plt.show()
pd.DataFrame(test_metrics["top_confusions"])

### 18.3 Validation vs test

In [ ]:
val_vs_test = pd.DataFrame({"validation": summary_row(transfer_val if final_version == transfer_version else baseline_val),
                            "test": summary_row(test_metrics)})
val_vs_test["difference"] = val_vs_test["test"] - val_vs_test["validation"]
val_vs_test

**Interpretation — generalisation.** On the 2,228 protected test crops the selected model reaches
**accuracy 0.837, macro precision 0.848, macro recall 0.838, macro-F1 0.837 and loss 0.515**. That is about
**8 points below validation** (0.920). This gap is the most important finding of the evaluation.

* **Systematic confusions:**
  - *Ascaris* → *Hymenolepis diminuta* (34) and *H. nana* → *H. diminuta* (30). *H. diminuta* has recall
    0.98 but precision 0.70, so the model over-uses it for roundish eggs.
  - *Paragonimus* → *Fasciolopsis* (27) and → *Ascaris* (25). *Paragonimus* recall is 0.71.
  - *Capillaria* → *Enterobius* (24). *Capillaria* is the weakest class (recall 0.69, F1 0.72).
  - *Trichuris* is never wrongly predicted (precision 1.00), but 21% of *Trichuris* eggs are missed.
* **Why lower than validation?** Validation photos come from the *same* official training collection as
  the training photos. Test photos are a separately released set that differs measurably:
  - It is darker and sharper on average (Section 8).
  - It has a different camera mix: 50% of test crops come from three high-resolution phone formats
    (vs 30% of training crops), and three formats seen in training are absent.
  - `scripts/analyze_test_errors.py` shows accuracy on those phone formats dropping from 0.92–0.95
    (validation) to 0.79–0.82 (test). The other formats stay at 0.83–0.90 on test.

  The gap is therefore mainly a **distribution shift between acquisition sets**. Leakage is ruled out by
  the audit, and it is not overfitting to the training images.
* **What it means:** validation accuracy overstates performance on new capture conditions. The test
  figure, **≈0.84**, is the honest estimate for this model. Over three complete runs the test macro-F1 was
  0.811, 0.821 and 0.837. Training on more diverse capture conditions would likely matter more than a
  larger network.

### 18.4 Overall performance summary

All seven metrics for both models on the validation set, and for the selected model on the held-out test
set. The baseline is **not** evaluated on the test set: the test set is used once, for the selected model
only. The second table puts the per-class results of the selected model on validation and test side by side.

In [ ]:
final_val = transfer_val if final_version == transfer_version else baseline_val
performance = pd.DataFrame({"Baseline CNN — validation": headline_row(baseline_val),
                            "MobileNetV2 — validation": headline_row(transfer_val),
                            f"Selected model ({final_version}) — test": headline_row(test_metrics)}).T
performance.to_csv(METRICS / "performance_summary.csv")

fig, ax = plt.subplots(figsize=(14, 4.8))
bar_metrics = METRIC_LABELS[:-1]   # log loss has a different scale and is shown in the table
x = np.arange(len(bar_metrics))
for i, (name, row) in enumerate(performance.iterrows()):
    bars = ax.bar(x + (i - 1) * 0.27, row[bar_metrics].astype(float), width=0.27, label=name)
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
ax.set_xticks(x, bar_metrics)
lowest = performance[bar_metrics].astype(float).min().min()
ax.set_ylim(max(0.0, np.floor((lowest - 0.05) * 10) / 10), 1.04)
ax.set_ylabel("score (higher is better)")
ax.set_title("Overall performance (log loss in the table below)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS / "performance_summary.png", dpi=120)
plt.show()
performance

In [ ]:
per_class_final = pd.concat({"Validation": per_class_frame(final_val), "Test": per_class_frame(test_metrics)}, axis=1)
per_class_final.to_csv(METRICS / "per_class_metrics_final_model.csv")

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), sharey=True)
for ax, metric in zip(axes, ["F1-score", "ROC-AUC", "PR-AUC"]):
    frame = pd.DataFrame({"validation": per_class_final[("Validation", metric)],
                          "test": per_class_final[("Test", metric)]}).astype(float)
    frame.plot.barh(ax=ax, width=0.8)
    ax.set_title(f"{metric} per class ({final_version})")
    ax.set_xlim(max(0.0, np.floor((frame.min().min() - 0.05) * 10) / 10), 1.0)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
    ax.grid(axis="x", alpha=0.3)
axes[0].invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS / "per_class_val_vs_test.png", dpi=120)
plt.show()
per_class_final.round(3)

**Interpretation.** _(filled in after execution)_

## Section 19 — Prediction demonstration

The **saved** model is reloaded through `Predictor` (below), the same class the API imports from `src/prediction.py`. It then
classifies randomly drawn, genuinely labelled test crops. The examples are random, not hand-picked, so
errors are shown if they occur. The second figure (made during evaluation) shows correct,
incorrect and lowest-confidence test predictions.

In [ ]:
# --- copied from src/prediction.py ---
DISCLAIMER = ("Confidence is the model's softmax score, not a probability of being correct. "
              "OvaScope is a student research prototype, not a validated diagnostic tool.")


class Predictor:
    """Holds one loaded model plus its class mapping and metadata."""

    def __init__(self, version_dir: Path):
        import tensorflow as tf
        version_dir = Path(version_dir)
        model_path = version_dir / "model.keras"
        if not model_path.exists():
            raise FileNotFoundError(f"No model found at {model_path}. Train one with `python -m src.train`.")
        self.model = tf.keras.models.load_model(model_path, compile=False)
        # Compile the forward pass once. Calling a Keras model eagerly dispatches its ~150 layers
        # one by one from Python (measured: ~340 ms/image on CPU for MobileNetV2 vs ~16 ms compiled).
        size = int(self.model.input_shape[1])
        self._forward = tf.function(lambda batch: self.model(batch, training=False),
                                    input_signature=[tf.TensorSpec([None, size, size, 3], tf.float32)])
        self.class_map = load_class_map(version_dir / "class_mapping.json")
        self.metadata = json.loads((version_dir / "model_metadata.json").read_text())
        self.version = self.metadata["version"]
        self.image_size = int(self.metadata["input_shape"][0])
        output_classes = self.model.output_shape[-1]
        if output_classes != len(self.class_map):
            raise ValueError(f"Model outputs {output_classes} classes but class mapping has {len(self.class_map)}")

    @classmethod
    def from_active(cls, cfg: dict) -> "Predictor":
        models_dir = cfg["paths"]["models_dir"]
        version = get_active_version(models_dir)
        if version is None:
            raise FileNotFoundError(f"No active model in {models_dir}. Train and activate a model first.")
        return cls(version_path(models_dir, version))

    @classmethod
    def from_version(cls, cfg: dict, version: str) -> "Predictor":
        return cls(version_path(cfg["paths"]["models_dir"], version))

    def predict_probabilities(self, image) -> np.ndarray:
        batch = preprocess_pil_image(image, self.image_size)
        return self._forward(batch).numpy()[0]

    def predict_image(self, image, top_k: int = 3) -> dict:
        probabilities = self.predict_probabilities(image)
        order = np.argsort(probabilities)[::-1][:top_k]
        best = int(order[0])
        entry = self.class_map[best]
        return {
            "predicted_class_id": best,
            "predicted_class": entry["display_name"],
            "class_key": entry["folder"][3:],          # e.g. "ascaris_lumbricoides"
            "original_label": entry["original_name"],
            "confidence": float(probabilities[best]),
            "top_k": [{"class_id": int(i), "class_name": self.class_map[i]["display_name"],
                       "confidence": float(probabilities[i])} for i in order],
            "model_version": self.version,
            "note": DISCLAIMER,
        }

    def predict_bytes(self, data: bytes, top_k: int = 3) -> dict:
        """Validate + decode raw file bytes, then predict. Raises ValueError for unreadable input."""
        return self.predict_image(load_image_from_bytes(data), top_k=top_k)

    def predict_file(self, path, top_k: int = 3) -> dict:
        return self.predict_bytes(Path(path).read_bytes(), top_k=top_k)

In [ ]:
predictor = Predictor.from_active(cfg)
demo_rows = manifest[manifest.split == "test"].sample(12, random_state=7)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
results = []
for ax, (_, row) in zip(axes.flat, demo_rows.iterrows()):
    result = predictor.predict_file(PROCESSED_DIR / row["crop_path"])
    correct = result["predicted_class_id"] == row["class_id"]
    results.append({"crop": row["crop_id"], "true": row["class_name"], "predicted": result["predicted_class"],
                    "confidence": round(result["confidence"], 3), "correct": correct})
    ax.imshow(Image.open(PROCESSED_DIR / row["crop_path"]))
    ax.set_title(f"true: {row['class_name']}\npred: {result['predicted_class']}\nconf {result['confidence']:.2f}",
                 fontsize=8, color="green" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Model version used:", predictor.version)
pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(16, 9.5))
plt.imshow(Image.open(PLOTS / "final_test_predictions.png"))
plt.axis("off")
plt.show()

**Interpretation.** The reloaded model correctly classified **9 of the 12 randomly drawn test crops**. The 3
errors are *Capillaria* → *Opisthorchis* (confidence 0.63), *Paragonimus* → *Hookworm* (0.54) and
*Paragonimus* → *Fasciolopsis* (0.60). All three had confidence ≤ 0.63, while 8 of the 9 correct predictions
scored above 0.95. In this sample a low score is a useful warning, but it is **not a guarantee**: one
correct prediction scored only 0.49, and the larger evaluation also contains confident errors (e.g.
*H. nana* → *Enterobius* at 0.95 in the second figure).

In the second figure, most incorrect and lowest-confidence examples are blurred, faint or dark eggs.
Two of the lowest-confidence ones, however, are sharp eggs (a *Trichuris* and a hookworm egg) on busy,
debris-filled backgrounds. Poor focus is one cause of uncertainty, but not the only one.

## Section 20 — Saving the model

Every version folder contains the native **`.keras`** model (the primary format) and an **`.h5`** export.
The `.h5` file was written with `model.save("model.h5")`, not renamed. Both files are reloaded below
and must give the same predictions. The folder also holds the class mapping, configuration, training
history, metadata (with SHA-256 checksums) and the model summary.

In [ ]:
for path in sorted(final_folder.iterdir()):
    print(f"{path.name:28s} {path.stat().st_size / 1e6:8.2f} MB")
sample = next(iter(make_dataset(test_files[:8], test_labels[:8], IMAGE_SIZE, 8, training=False)))[0]
reloaded_keras = tf.keras.models.load_model(final_folder / "model.keras", compile=False)
reloaded_h5 = tf.keras.models.load_model(final_folder / "model.h5", compile=False)
difference = np.abs(reloaded_keras.predict(sample, verbose=0) - reloaded_h5.predict(sample, verbose=0)).max()
print("Max difference between .keras and .h5 predictions:", difference)
print("Active model pointer:", json.loads((cfg["paths"]["models_dir"] / "active_model.json").read_text()))

## Section 21 — Retraining readiness

The application retrains by **loading the saved OvaScope model** and continuing training. It never
starts from a randomly initialised network. The full process lives in `src/retrain.py`:

1. Labelled images are uploaded and validated, then stored as BLOBs in SQLite (`src/database.py`).
2. *Retrain* loads the active model version and builds a training set from the new images plus a
   **replay sample** of original training crops (this limits *catastrophic forgetting*).
3. Training continues for a few epochs at a small learning rate (1e-5).
4. The candidate and the active model are evaluated on the validation set (the *promotion set*).
5. The candidate is saved as a new version. It is activated only if its macro-F1 is not worse than the
   active model's (within the tolerance); the old version stays available for rollback.

The controlled demonstration below continues training a reloaded copy for one epoch on a small replay
sample. It shows that the saved model reloads, accepts new batches, and that its weights change. Nothing
is saved, so the activated model is not modified.

In [ ]:
# --- copied from src/retrain.py (replay sample used by retraining) ---
def replay_sample(processed_dir: Path, per_class: int, seed: int) -> tuple[list[str], list[int]]:
    files, labels = list_split(processed_dir, "train")
    by_class: dict[int, list[str]] = {}
    for f, y in zip(files, labels):
        by_class.setdefault(y, []).append(f)
    rng = random.Random(seed)
    out_files, out_labels = [], []
    for y in sorted(by_class):
        chosen = rng.sample(by_class[y], min(per_class, len(by_class[y])))
        out_files += chosen
        out_labels += [y] * len(chosen)
    return out_files, out_labels

In [ ]:
replay_files, replay_labels = replay_sample(PROCESSED_DIR, per_class=10, seed=SEED)
continued = tf.keras.models.load_model(final_folder / "model.keras", compile=False)
continued.compile(optimizer=tf.keras.optimizers.Adam(cfg["retraining"]["learning_rate"]),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
weights_before = [w.numpy().copy() for w in continued.trainable_weights]
history = continued.fit(make_dataset(replay_files, replay_labels, IMAGE_SIZE, BATCH_SIZE, training=True, seed=SEED),
                        epochs=1, verbose=2)
changed = sum(not np.allclose(a, b.numpy()) for a, b in zip(weights_before, continued.trainable_weights))
print(f"Continued from {final_version}: {changed}/{len(weights_before)} trainable weight tensors changed")

## Section 22 — Exporting results

All generated artifacts are packaged into **`ovascope_training_results.zip`**. Only files that really
exist are added; nothing is padded with placeholders.

* **Local:** the archive is in `artifacts/exports/`.
* **Colab:** the next cell offers a browser download, and the archive is also on Drive if `USE_GOOGLE_DRIVE` is set.
* **Kaggle:** files in `/kaggle/working` are **not** transferred automatically. Click *Save Version*
  (*Save & Run All*), then download the ZIP from the version's **Output** tab, or download it straight
  from the right-hand *Output* panel while the session is running.

In [ ]:
from scripts.export_artifacts import build_results_zip

results_zip = build_results_zip(cfg)
import zipfile as _zip
with _zip.ZipFile(results_zip) as archive:
    for info in archive.infolist():
        print(f"{info.filename:36s} {info.file_size / 1e6:8.2f} MB")
print("\nArchive location:", results_zip)
if ENV == "colab":
    from google.colab import files
    files.download(str(results_zip))

## Section 23 — Final discussion

**What was achieved**
* The official detection dataset was turned into a verified 11-class crop dataset. It rebuilds
  byte-identically, has no source-photo or near-duplicate leakage across splits, and keeps the official
  labels.
* A baseline CNN and a MobileNetV2 transfer model were trained with augmentation, dropout, early stopping
  (best-weight restoration), LR scheduling, checkpointing and fine-tuning, then compared on validation data.
* The selected model was evaluated once on the protected test set, saved as `.keras` and `.h5` (reload
  verified identical), and made ready for retraining.

**Which model performed best:** MobileNetV2 with fine-tuning. Validation macro-F1 0.920, versus 0.860 for
the baseline in this run.

**What the results actually show**
* On the official test set: **accuracy 0.837, macro-F1 0.837** (three runs: 0.811 / 0.821 / 0.837).
* It is strongest on *Taenia* (F1 0.96), *Hookworm* (0.91) and *Fasciolopsis* (0.89).
* It is weakest on *Capillaria* (0.72), *Ascaris* (0.74) and *H. nana* (0.78).
* It over-predicts *H. diminuta*.

**Limitations**
* About an 8-point validation-to-test drop, linked to differences in camera/acquisition between the
  official training and test collections.
* The input must be a crop of one egg; there is no detection.
* Absolute egg size is lost by resizing.
* Confidence scores are not calibrated probabilities.
* Single-run numbers vary by a few points between identical runs, because of GPU non-determinism.

**Potential improvements**
1. Validate on photos *grouped by capture device/session*, so validation predicts new-device performance.
2. Use stronger colour/illumination augmentation or stain/colour normalisation.
3. Give the model the crop's physical scale as an extra input.
4. Add a detection stage (the bounding boxes are already available).
5. Calibrate confidence scores.
6. Average several training runs.

Any such change must be selected on validation data, not on the test set used here.

## Section 24 — Conclusion

**Completed in this notebook**
* data verification and reproducible crop generation from the official Zenodo files;
* exploratory analysis with three interpreted image features;
* reusable preprocessing with training-only augmentation;
* a baseline CNN and a fine-tuned MobileNetV2, compared on validation data;
* a single final test evaluation: accuracy 0.837, macro-F1 0.837;
* model saving in `.keras` and `.h5` with metadata and checksums;
* a prediction demonstration with real errors shown;
* a retraining-readiness check;
* the exported results archive.

**Outside this notebook (in the application code)**
* the FastAPI service, SQLite uploads and real retraining jobs with promotion (`src/retrain.py`);
* the Streamlit interface, monitoring, Docker and load testing — see `README.md` and `reports/`.

**Future work**
* device-aware validation, colour/stain normalisation and calibrated confidence (see Section 23);
* an egg-detection stage, so full microscope photos can be processed.